# RFEID is who has cited a listed paper!


I would like to share that when using the COMPLETE view the maximum results retrieve is 25 per API call.

In order to get the remaining records, you can use the 'start' parameter. Numeric value representing the results offset (i.e. starting position for the search results). The maximum for this value is a system-level default (varies with search cluster) minus the number of results requested. If not specified the offset will be set to zero (i.e. first search result)

ex. start=5

If you will use the STANDARD view, we can increase the number of records return per API call by using the 'count' parameter. The maximum value of count -\is 200. In addition the number cannot exceed the maximum system default - if it does an error will be returned.



In [ ]:
# DATA ORGANIZATION
# references_json_1 to 6 are in snowball -3., i.e. these are in the 'core sample' Same is true for cited_by_json_1 till 3_3. All others are beyond this sample.
# This means that the papers beyond this core sample should just be used for it's meta-data, and with no references and citations of it's own.

In [ ]:
# Version STANDARD for up to 200 references
f'https://api.elsevier.com/content/search/scopus?apiKey=XXX&view=STANDARD&count=200&httpAccept=application/json&query=REFEID({XXXXXXX})'
# Version COMPLETE. Gets you 25 references per page. Starts with page 0 with start at 0. Page 1 starts with index 24
f'Version for up to 200 referenceshttps://api.elsevier.com/content/search/scopus?apiKey=YYY&view=COMPLETE&start={NO}&httpAccept=application/json&query=REFEID({XXXXXX})'

In [ ]:
%pwd

In [ ]:
# https://www.geeksforgeeks.org/how-to-run-deepseek-r1-locally-free-mac-windows-linux-guide/
# !git clone https://huggingface.co/deepseek-ai/deepseek-r1
# from transformers import AutoModelForCausalLM, AutoTokenizer

# https://api.elsevier.com/content/abstract/eid/2-s2.0-86000131388?apiKey=XXX
# https://api.elsevier.com/content/search/scopus?apiKey=XXX&view=COMPLETE&httpAccept=application/json&suppressNavLinks=true&query=REFEID(2-s2.0-84864329732)&field=eid,prism:

In [ ]:
!pip install humanfriendly
!pip install habanero  # Install the habanero package
!pip install doi2bib # https://pypi.org/project/doi2bib/
!pip install pybliometrics
!pip install torch transformers accelerate
!pip install transformers
!pip install torch
!pip install tensorflow
!pip install sentence_transformers
!pip install bert-extractive-summarizer
!pip install spacy==2.0.12
!pip install cairocffi
!pip install summarizer
!pip install flashtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=b828cd114e628e0b8452685602e5b1555923b6888b39fec0d1592aa09ab99349
  Stored in directory: /root/.cache/pip/wheels/54/f8/e6/ecfceb6af875ddc5096bb3811795ac336f50371009a601454d
Successfully built bibtexparser
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.5/126.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 7.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while genera

In [ ]:
import requests
import pandas as pd
import json
import re
# https://github.com/allenai/s2-folks/tree/main/examples/python
# Cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import datetime
from datetime import datetime, date, time, timedelta
import http.client, urllib
import time
from time import strftime
from time import gmtime
from humanfriendly import format_timespan
from pybliometrics.scopus import CitationOverview
import requests
from habanero import Crossref
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
import pybliometrics
drive.mount('/content/drive')
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review")  # set WD
import time
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import linkage
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from scipy.spatial.distance import squareform # Corrected import
from scipy.cluster.hierarchy import cut_tree, dendrogram, linkage #Import necessary functions
# from summarizer import Summarizer,TransformerSummarizer
from transformers import BartForConditionalGeneration, BartTokenizer
from collections import defaultdict
import zipfile
from habanero import Crossref
from collections import defaultdict
import pickle
from collections import defaultdict
from flashtext import KeywordProcessor


Mounted at /content/drive


In [ ]:
# prompt: count number of files in /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info

import os

path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+1_cited_by"
num_files = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
print(f"Number of files in '{path}': {num_files}")

Number of files in '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+1_cited_by': 2431


In [1]:
import http.client, urllib
from google.colab import userdata

def send_push_final(total_elapsed_time):
    token = userdata.get('PUSHOVER_TOKEN')
    user = userdata.get('PUSHOVER_USER')
    if not token or not user:
        print("Missing Pushover credentials in Secrets.")
        return

    conn = http.client.HTTPSConnection("api.pushover.net:443")
    conn.request("POST", "/1/messages.json",
      urllib.parse.urlencode({
        "token": token,
        "user": user,
        "title": "Colab: -  Total job done - data download",
        "message": f"Total elapsed time = {total_elapsed_time}",
      }), { "Content-type": "application/x-www-form-urlencoded" })
    conn.getresponse()

In [ ]:
FPE_core = pd.read_csv('FPE_core.csv')

In [ ]:
def get_scopus_urls_STANDARD(df, api_key): # search field used to identify all articles that cite a specific publication
    scopus_urls = []
    for scopus_code in df:
        url = f'https://api.elsevier.com/content/search/scopus?apiKey={api_key}&view=STANDARD&count=200&httpAccept=application/json&query=REFEID({scopus_code})'
        scopus_urls.append(url)
    return scopus_urls

def get_scopus_urls_COMPLETE(expanded_table, api_key): # search field used to identify all articles that cite a specific publication
    scopus_urls = []
    for i  in  range(len(expanded_table)):
        row = expanded_table[i:i+1]
        REFEID = row['search_term'].values[0]
        NO = row['rounded_up_value'].values[0]
        url = f'https://api.elsevier.com/content/search/scopus?apiKey={api_key}&view=COMPLETE&start={NO}&httpAccept=application/json&query=REFEID({REFEID})'
        scopus_urls.append(url)
    return scopus_urls

def list_files(file_path):
  if os.path.exists(file_path):
    return [f for f in os.listdir(file_path) if os.path.isfile(os.path.join(file_path, f))]
  else:
    return None

def open_and_read_json(filepath):
  try:
    with open(filepath, 'r', encoding='utf-8') as file:
      return json.load(file)
  except UnicodeDecodeError:
    print(f"UnicodeDecodeError: Retrying with 'latin-1' for '{filepath}'.")
    try:
      with open(filepath, 'r', encoding='latin-1') as file:
        return json.load(file)
    except Exception as e:
      print(f"Error: Could not read '{filepath}' with 'latin-1' either: {e}")
      return None
  except FileNotFoundError:
    print(f"Error: File '{filepath}' not found.")
    return None
  except json.JSONDecodeError:
    print(f"Error: Invalid JSON format in '{filepath}'.")
    return None
  except Exception as e:
    print(f"An unexpected error occurred while reading '{filepath}': {e}")
    return None

# Iterate through json_all_data and find the dictionary where the search term matches

def find_matching_data(json_all_data, search_term):
    matching_data = None
    for data_item in json_all_data:
        try:
            search_terms = data_item['search-results']['opensearch:Query']['@searchTerms']
            if search_terms == search_term:
                matching_data = data_item
                break # Stop when the first match is found
        except (KeyError, TypeError):
            # Handle cases where the expected keys might be missing
            continue
    return matching_data



def get_scopus_urls_ABSTRACT(df, api_key): # search field used to identify all references that are listed in the specified publication
    scopus_urls = []
    for DOI in df:
        url = f'https://api.elsevier.com/content/abstract/doi/{DOI}?apiKey={api_key}&httpAccept=application/json'
        scopus_urls.append(url)
    return scopus_urls



# Helper function to safely get nested values (not strictly necessary with the current implementation, but good practice)
def _safe_get(data, keys, default=None):
    if not isinstance(keys, list):
        keys = [keys]
    current = data
    for key in keys:
        if isinstance(current, dict) and key in current:
            current = current[key]
        elif isinstance(current, list) and isinstance(key, int) and len(current) > key:
            current = current[key]
        else:
            return default
    return current


api_key = 'XYZ' # Replace with your actual API key

In [ ]:
https://api.elsevier.com/content/abstract/doi/10.1016/j.ijdrr.2020.101596?apiKey=YYY&httpAccept=application/json

In [ ]:
https://api.elsevier.com/content/abstract/doi/10.1016/j.forpol.2017.03.007?apiKey=YYY&httpAccept=application/json

In [ ]:
# Make a list of URLs based on a list of scopus_code

# STANDARD VIEW - UP TO 200 CITATIONS

# Assuming FPE_core is your DataFrame and your API key is 'your_api_key'
api_key = 'YYY' # Replace with your actual API key
scopus_links = get_scopus_urls(FPE_core, api_key)
print(scopus_links[:5]) # Print the first 5 links as a sample

scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})
scopus_links_df.to_csv('scopus_links.csv', index=False)


In [ ]:
# load all the files from file_path as json and then combine all of them into a single list

# Example usage (replace with your desired file path):
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/scopus_urls_missing_abstract_references"
files = list_files(file_path)

# Initialize json_all_data as a list to store the combined data
json_all_data_1 = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_1.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

In [ ]:
len(json_all_data_1)

2289

In [ ]:
with open('scopus_urls_missing_abstract_references.pkl', 'wb') as f:
  pickle.dump(json_all_data_1, f)

In [ ]:
json_all_data_Snowball_7_2 = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_7_2.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

In [ ]:
len(json_all_data_Snowball_7_2)

3124

In [ ]:
# IDENTIFY REFEID OF PAPERS THAT HAVE BEEN CITED MORE THAN 200 TIMES
# cycle through json_all_data and get count in json_all_data[i]['search-results']['opensearch:totalResults'] per  json_all_data[i]

total_results_counts = []
for i in range(len(json_all_data)):
    try:
        count = int(json_all_data[i]['search-results']['opensearch:totalResults'])
        total_results_counts.append(count)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not extract count for item at index {i}: {e}")
        total_results_counts.append(None) # Append None or handle the error as needed

# You can now use the total_results_counts list, for example:
print("Total results counts per JSON object:")
total_results_counts

# in json_all_data, identify all json_all_data[i]['search-results']['opensearch:Query']['@searchTerms'] for which json_all_data[i]['search-results']['opensearch:totalResults'] is larger than 199

terms_with_many_results = []
for i in range(len(json_all_data)):
    try:
        total_results = int(json_all_data[i]['search-results']['opensearch:totalResults'])
        search_terms = json_all_data[i]['search-results']['opensearch:Query']['@searchTerms']
        if total_results > 199:
            terms_with_many_results.append(search_terms)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not process item at index {i}: {e}")

print("Search terms with more than 199 total results:")
terms_with_many_results

# Prepare df to make a file for bulk-download of papers that have been cited more than 200 times

table_data = [] # Create an empty list to store data for the table

for i in range(len(terms_with_many_results)):
    search_term = terms_with_many_results[i]
    matching_data = find_matching_data(json_all_data, search_term)
    total_results_str = matching_data['search-results']['opensearch:totalResults']
    total_results = int(total_results_str)
    rounded_up_value = int(np.ceil(total_results / 25))
    table_data.append({'search_term': search_term, 'total_results': total_results, 'rounded_up_value': rounded_up_value})


# After the loop, create the DataFrame from the list of dictionaries
table = pd.DataFrame(table_data)
table['search_term'] = table['search_term'].str.replace('REFEID', '').str.replace('(', '').str.replace(')', '')

expanded_rows = []

for index, row in table.iterrows():
    search_term = row['search_term']
    rounded_up_value = row['rounded_up_value']
    for i in range(rounded_up_value):
        expanded_rows.append({'search_term': search_term, 'rounded_up_value': rounded_up_value})

expanded_table = pd.DataFrame(expanded_rows)
expanded_table = expanded_table.assign(rounded_up_value=expanded_table.groupby('search_term').cumcount())

# Make a list of URLs based on a list of scopus_code

# COMPLETE VIEW - 25 CITATIONS PER PAGE

# Assuming FPE_core is your DataFrame and your API key is 'your_api_key'
api_key = 'XYZ' # Replace with your actual API key
scopus_links = get_scopus_urls(expanded_table, api_key)
print(scopus_links[:5]) # Print the first 5 links as a sample

In [ ]:
scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})
scopus_links_df.to_csv('scopus_links.csv', index=False)

In [ ]:
scopus_links_df.to_csv('scopus_links_1.csv', index=False)

In [ ]:
# MAKE A SINGLE LIST FROM ALL PAPERS IN SNOWBALL 1
# PAPERS THAT ARE CITED MANY TIMES ARE PRESENT IN MULTIPLE ELEMENTS OF THE LIST, BUT ALL OF THEIR 'CITED BY' ARE ACCOUNTED FOR
# Now combine all the files from Snowball+1_long
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+1_long_cited_by"
files = list_files(file_path)

# Initialize json_all_data as a list to store the combined data
json_all_data_Snowball_1_long = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_1_long.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

# remove elements in json_all_data for which json_all_data[i]['search-results']['opensearch:Query']['@searchTerms'] that are in  terms_with_many_results

json_all_data_filtered = [
    data for data in json_all_data
    if 'search-results' in data and
       'opensearch:Query' in data['search-results'] and
       '@searchTerms' in data['search-results']['opensearch:Query'] and
       data['search-results']['opensearch:Query']['@searchTerms'] not in terms_with_many_results
]

#  join two lists: json_all_data_filtered and json_all_data_Snowball_1_long
json_combined_data = json_all_data_filtered + json_all_data_Snowball_1_long
len(json_combined_data)

In [ ]:
# make  basic_paper_data and edgelist_0

# FIRST FILE: basic_paper_data: Title DOI Abstract EID
# SECOND FILE: edgelist_0:     (JUST EIDs AND DOIs)

FPE_core = pd.read_csv('FPE_core.csv')
basic_paper_data = FPE_core[['EID', 'DOI', 'Title', 'Abstract']]
edgelist_0 = pd.DataFrame(
    columns=['source_EID', 'source_DOI', 'target_EID', 'target_DOI'])


for i in range(0, len(json_combined_data)):
    search_EID = json_combined_data[i]['search-results']['opensearch:Query'][
        '@searchTerms']  # searchTerms
    search_EID = search_EID.replace("REFEID(", "").replace(")", "")
    search_doi = None
    for index, row in basic_paper_data.iterrows():
        if row['EID'] == search_EID:
            search_doi = row['DOI']
            break
    for j in range(0, len(json_combined_data[i]['search-results']['entry'])):
        # Check if 'eid' key exists before accessing it
        entry = json_combined_data[i]['search-results']['entry'][j]
        if 'eid' in entry:
            one_eid = entry['eid']
        else:
            # Handle the case where 'eid' is missing, e.g., skip this entry or assign a default value
            print(f"Warning: 'eid' key missing in entry {j} of search results {i}. Skipping this entry.")
            continue  # Skip to the next entry

        # Check if 'prism:doi' key exists before accessing it
        one_doi = entry.get('prism:doi')
        # If 'prism:doi' is not found, assign None or an empty string
        if one_doi is None:
            one_doi = ''  # or one_doi = None
        one_title = entry['dc:title']
        # Check if 'dc:description' key exists before accessing it
        one_abstract = entry.get('dc:description', '')  # Assign empty string if not found
        one_paper_data = pd.DataFrame({
            'EID': [one_eid],
            'DOI': [one_doi],
            'Title': [one_title],
            'Abstract': [one_abstract]
        })
        if one_doi not in basic_paper_data['DOI'].values:  # add one_paper_data to basic_paper_data only if one_doi is not in basic_paper_data['DOI']
            basic_paper_data = pd.concat([basic_paper_data, one_paper_data],
                                         ignore_index=True)
        one_paper_edgelist = pd.DataFrame({
            'source_EID': [one_eid],
            'source_DOI': [one_doi],
            'target_EID': [search_EID], #
            'target_DOI': [search_doi] #
        })
        edgelist_0 = pd.concat([edgelist_0, one_paper_edgelist], ignore_index=True)

In [ ]:
# Here define top-level communities - to separate those that are completley out of FPE

In [ ]:
edgelist_0.to_csv('edgelist_snowball_1.csv')
basic_paper_data.to_csv('basic_paper_data_snowball_1.csv')

In [ ]:
edgelist_0 = pd.read_csv('edgelist_snowball_1.csv')
basic_paper_data = pd.read_csv('basic_paper_data_snowball_1.csv')
edgelist_0 = edgelist_0.drop(columns=['Unnamed: 0'])
edgelist_0 = edgelist_0.drop(columns=['target_EID'])
edgelist_0 = edgelist_0.drop(columns=['target_DOI'])
edgelist_0 = edgelist_0[~edgelist_0['source_DOI'].str.contains('j.forpol', na=False)] #  from edgelist_0, remove all rows where entries in column source_DOI contain j.forpol
edgelist_0 = edgelist_0.drop_duplicates() # remove duplicate rows from edgelist_0


In [ ]:
# Now get the initial EIDs for Snowball +2


# Assuming FPE_core is your DataFrame and your API key is 'your_api_key'
api_key = 'XYZ' # Replace with your actual API key
scopus_links = get_scopus_urls(edgelist_0['source_EID'], api_key)
print(scopus_links[:5]) # Print the first 5 links as a sample

scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})

# save scopus_links_df top csv - with max 10000 rows per file

max_rows_per_file = 10000
num_files = len(scopus_links_df) // max_rows_per_file + (len(scopus_links_df) % max_rows_per_file != 0)

for i in range(num_files):
    start_row = i * max_rows_per_file
    end_row = min((i + 1) * max_rows_per_file, len(scopus_links_df))
    df_subset = scopus_links_df.iloc[start_row:end_row]
    file_name = f'scopus_links_{i+1}.csv'
    df_subset.to_csv(file_name, index=False)

In [ ]:
# Unzip the Snowball+2
with zipfile.ZipFile("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+2.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review")

In [ ]:
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+2"
files = list_files(file_path)

In [ ]:
# Now combine all the files from Snowball+2
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+2"
files = list_files(file_path)

# Initialize json_all_data as a list to store the combined data
json_all_data_Snowball_2 = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_2.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

In [ ]:
# Identify papers that have more that have been cited by more than 199 times -
terms_with_many_results = []
for i in range(len(json_all_data_Snowball_2)):
    try:
        total_results = int(json_all_data_Snowball_2[i]['search-results']['opensearch:totalResults'])
        search_terms = json_all_data_Snowball_2[i]['search-results']['opensearch:Query']['@searchTerms']
        if total_results > 199:
            terms_with_many_results.append(search_terms)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not process item at index {i}: {e}")

print("Search terms with more than 199 total results:")
len(terms_with_many_results)

Search terms with more than 199 total results:


346

In [ ]:
# Identify papers that have more that have been cited by more than 199 times -
terms_with_many_results = []
for i in range(len(json_all_data_Snowball_2)):
    try:
        total_results = int(json_all_data_Snowball_2[i]['search-results']['opensearch:totalResults'])
        search_terms = json_all_data_Snowball_2[i]['search-results']['opensearch:Query']['@searchTerms']
        if total_results > 199:
            terms_with_many_results.append(search_terms)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not process item at index {i}: {e}")

print("Search terms with more than 199 total results:")
terms_with_many_results

table_data = [] # Create an empty list to store data for the table

for i in range(len(terms_with_many_results)):
    search_term = terms_with_many_results[i]
    matching_data = find_matching_data(json_all_data_Snowball_2, search_term)
    total_results_str = matching_data['search-results']['opensearch:totalResults']
    total_results = int(total_results_str)
    rounded_up_value = int(np.ceil(total_results / 25))
    table_data.append({'search_term': search_term, 'total_results': total_results, 'rounded_up_value': rounded_up_value})


# After the loop, create the DataFrame from the list of dictionaries
table = pd.DataFrame(table_data)
table['search_term'] = table['search_term'].str.replace('REFEID', '').str.replace('(', '').str.replace(')', '')

expanded_rows = []

for index, row in table.iterrows():
    search_term = row['search_term']
    rounded_up_value = row['rounded_up_value']
    for i in range(rounded_up_value):
        expanded_rows.append({'search_term': search_term, 'rounded_up_value': rounded_up_value})

expanded_table = pd.DataFrame(expanded_rows)
expanded_table = expanded_table.assign(rounded_up_value=expanded_table.groupby('search_term').cumcount())

scopus_links = get_scopus_urls_COMPLETE(expanded_table, api_key)
scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})
scopus_links_df.to_csv('scopus_links_2_long.csv', index=False)

Search terms with more than 199 total results:


In [ ]:
expanded_table = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/expanded_table_long_snowball_cited_by_3.csv')
expanded_table = expanded_table.drop(columns=['Unnamed: 0'])

In [ ]:
api_key = 'XYZ'
scopus_links = get_scopus_urls_COMPLETE(expanded_table, api_key)
scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})
scopus_links_df.to_csv('scopus_links_3_long.csv', index=False)

In [ ]:
scopus_links_df.to_csv('scopus_links_3_long.csv', index=False)

In [ ]:
# Unzip the Snowball+2
with zipfile.ZipFile("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+2_long.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review")

In [ ]:
# Now combine all the files from Snowball+2_long
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+2_long"
files = list_files(file_path)

# Initialize json_all_data as a list to store the combined data
json_all_data_Snowball_2_long = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_2_long.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

In [ ]:
len(json_all_data_Snowball_2_long)

In [ ]:
# remove elements in json_all_data for which json_all_data[i]['search-results']['opensearch:Query']['@searchTerms'] that are in  terms_with_many_results
json_all_data_filtered = [
    data for data in json_all_data_Snowball_2
    if 'search-results' in data and
       'opensearch:Query' in data['search-results'] and
       '@searchTerms' in data['search-results']['opensearch:Query'] and
       data['search-results']['opensearch:Query']['@searchTerms'] not in terms_with_many_results
]

#  join two lists: json_all_data_filtered and json_all_data_Snowball_1_long
json_combined_data_Snowball_2 = json_all_data_filtered + json_all_data_Snowball_2_long
len(json_combined_data_Snowball_2)

38074

In [ ]:
basic_paper_data_snowball_1 = pd.read_csv('basic_paper_data_snowball_1.csv')
basic_paper_data_snowball_1 = basic_paper_data_snowball_1.drop(columns=['Unnamed: 0'])

In [ ]:
# make  basic_paper_data_2 and edgelist_2

# FIRST FILE: basic_paper_data_2: Title DOI Abstract EID
# SECOND FILE: edgelist_2:     (JUST EIDs AND DOIs)

basic_paper_data_2 = pd.DataFrame(columns = ['EID', 'DOI', 'Title', 'Abstract'])
edgelist_2 = pd.DataFrame(
    columns=['source_EID', 'source_DOI', 'target_EID', 'target_DOI'])


for i in range(0, len(json_combined_data_Snowball_2)):
    search_EID = json_combined_data_Snowball_2[i]['search-results']['opensearch:Query'][
        '@searchTerms']  # searchTerms
    search_EID = search_EID.replace("REFEID(", "").replace(")", "")
    search_doi = None
    for index, row in basic_paper_data_snowball_1.iterrows():
        if row['EID'] == search_EID:
            search_doi = row['DOI']
            break
    for j in range(0, len(json_combined_data_Snowball_2[i]['search-results']['entry'])):
        # Check if 'eid' key exists before accessing it
        entry = json_combined_data_Snowball_2[i]['search-results']['entry'][j]
        if 'eid' in entry:
            one_eid = entry['eid']
        else:
            # Handle the case where 'eid' is missing, e.g., skip this entry or assign a default value
            print(f"Warning: 'eid' key missing in entry {j} of search results {i}. Skipping this entry.")
            continue  # Skip to the next entry

        # Check if 'prism:doi' key exists before accessing it
        one_doi = entry.get('prism:doi')
        # If 'prism:doi' is not found, assign None or an empty string
        if one_doi is None:
            one_doi = ''  # or one_doi = None
        one_title = entry['dc:title']
        # Check if 'dc:description' key exists before accessing it
        one_abstract = entry.get('dc:description', '')  # Assign empty string if not found
        one_paper_data = pd.DataFrame({
            'EID': [one_eid],
            'DOI': [one_doi],
            'Title': [one_title],
            'Abstract': [one_abstract]
        })
        if one_doi not in basic_paper_data_2['DOI'].values:  # add one_paper_data to basic_paper_data only if one_doi is not in basic_paper_data['DOI']
            basic_paper_data_2 = pd.concat([basic_paper_data_2, one_paper_data],
                                         ignore_index=True)
        one_paper_edgelist = pd.DataFrame({
            'source_EID': [one_eid],
            'source_DOI': [one_doi],
            'target_EID': [search_EID], #
            'target_DOI': [search_doi] #
        })
        edgelist_2 = pd.concat([edgelist_2, one_paper_edgelist], ignore_index=True)

In [ ]:
#### basic_paper_data_1 and basic_paper_data_2 don't have abstracts of all papers - get them!
# This is because STANDARD search by RFEID does not give abstracts - only COMPLETE DOES.
# So now use Abstract search API by EID to get the abstracts for all papers in basic_paper_data_1 and basic_paper_data_2
basic_paper_data_2.to_csv('basic_paper_data_2.csv')
edgelist_2.to_csv('edgelist_2.csv')

In [ ]:
# NOW PREPARE LIST OF URLS TO DOWNLOAD ALL 'CITED BY' SNOWBALL +3 PAPERS IN STANDARD VIEW
# Load previous edgelists
edgelist_1 = pd.read_csv('edgelist_snowball_1_cited_by.csv')
edgelist_1 = edgelist_1.drop(columns=['Unnamed: 0'])
edgelist_1 = edgelist_1['source_EID']
edgelist_2 = pd.read_csv('edgelist_snowball_2_citated_by.csv')
edgelist_2 = edgelist_2.drop(columns=['Unnamed: 0'])
edgelist_2 = edgelist_2['source_EID']
# remove from edgelist_2 all elements that are in edgelist_1
edgelist_1_set = set(edgelist_1)
edgelist_3 = edgelist_2[~edgelist_2.isin(edgelist_1_set)]
edgelist_3 = edgelist_3.drop_duplicates()

api_key = 'XYZ' # Replace with your actual API key
scopus_links = get_scopus_urls_STANDARD(edgelist_3, api_key)
print(scopus_links[:5]) # Print the first 5 links as a sample

scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})

# save scopus_links_df top csv - with max 10000 rows per file

max_rows_per_file = 100000
num_files = len(scopus_links_df) // max_rows_per_file + (len(scopus_links_df) % max_rows_per_file != 0)

for i in range(num_files):
    start_row = i * max_rows_per_file
    end_row = min((i + 1) * max_rows_per_file, len(scopus_links_df))
    df_subset = scopus_links_df.iloc[start_row:end_row]
    file_name = f'scopus_links_snowball_cited_by_3_standard{i+1}.csv'
    df_subset.to_csv(file_name, index=False)

In [ ]:
# remove from edgelist_2 those which are in edgelist_1


# combining lists of 'cited by'

In [ ]:
# combining lists of 'cited by'
# Same is for +2 and +3 networks

# RUN THIS AGAIN AND CHECK IF cited_by and long_cited_by HAVE THE SAME DATA STRUCTURE!!!!!!!!!

file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+1_cited_by"
files = list_files(file_path)

json_all_data_Snowball_1 = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_1.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

# load all the files snowball + 1 long
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+1_long_cited_by"
files = list_files(file_path)
json_all_data_Snowball_1_long = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_1_long.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")



# Find all the papers that have been cited more than 199 times
terms_with_many_results = []
for i in range(len(json_all_data_Snowball_1)):
    try:
        total_results = int(json_all_data_Snowball_1[i]['search-results']['opensearch:totalResults'])
        search_terms = json_all_data_Snowball_1[i]['search-results']['opensearch:Query']['@searchTerms']
        if total_results > 199:
            terms_with_many_results.append(search_terms)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not process item at index {i}: {e}")

print("Search terms with more than 199 total results:")

# remove from json_all_data_Snowball_1 all papers that have been cited more than 199 times
json_all_data_filtered = [
    data for data in json_all_data_Snowball_1
    if 'search-results' in data and
       'opensearch:Query' in data['search-results'] and
       '@searchTerms' in data['search-results']['opensearch:Query'] and
       data['search-results']['opensearch:Query']['@searchTerms'] not in terms_with_many_results
]

#  join two lists: json_all_data_filtered and json_all_data_Snowball_1_long
json_cited_by_Snowball_1 = json_all_data_filtered + json_all_data_Snowball_1_long
len(json_cited_by_Snowball_1)


with open('json_cited_by_Snowball_1.pkl', 'wb') as f:
  pickle.dump(json_cited_by_Snowball_1, f)

Search terms with more than 199 total results:


In [ ]:
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/scopus_urls_df_for_snowball_2_citations"
files = list_files(file_path)

# Initialize json_citations_Snowball_2 as a list to store the combined data
json_citations_Snowball_2 = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    # Check if json_data is not None before appending
    if json_data is not None:
        json_citations_Snowball_2.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

print(f"Loaded {len(json_citations_Snowball_2)} JSON files.")

# Save the combined data to a pickle file for later use
with open('Snowball+2_citations_Single_paper_info.pkl', 'wb') as f:
    pickle.dump(json_citations_Snowball_2, f)
print("Successfully saved 'Snowball+2_citations_Single_paper_info.pkl'")

In [ ]:
with open('json_cited_by_Snowball_1.pkl', 'wb') as f:
  pickle.dump(json_cited_by_Snowball_1, f)

# CHUNK ON GETTING 'SNOWBALL_CITATIONS' DATA, I.E. DATA ON PAPERS THAT HAS INFORMATION WHICH PAPERS WERE CITED BY THE FOCAL PAPER (FORWARD SNOWBALL)

In [ ]:
####### GETTING DATA PER PAPER

# SAVE ALL DOWNLOAD LINKS AS TXT FILE - NOT CSV! IT'S FASTER!
# data.to_csv('scopus_links_snowball_cited_by_3_standard3.txt', sep='\t', index=False, header=False)

In [ ]:
FPE_core = pd.read_csv('FPE_core.csv')
scopus_urls = get_scopus_urls_ABSTRACT(FPE_core['DOI'], api_key)
scopus_urls_df = pd.DataFrame({'scopus_urls': scopus_urls})
scopus_urls_df.to_csv('scopus_urls_abstract.csv', index = False)

In [ ]:
# Unzip the Abstract data
with zipfile.ZipFile("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info_download_1.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/drive/MyDrive/Colab Notebooks/Forest_policy_review")

In [ ]:
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info"
files = list_files(file_path)

In [ ]:
len(files)

2425

In [ ]:
# Initialize json_all_data as a list to store the combined data
json_all_data_Snowball_1_citations = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_1_citations.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

In [ ]:
len(json_all_data_Snowball_1_citations)

2425

In [ ]:
# json_all_data_Snowball_1_citations[0]['abstracts-retrieval-response']['item']['bibrecord']['head']['citation-title'] # TITLE
# json_all_data_Snowball_1_citations[0]['abstracts-retrieval-response']['item']['bibrecord']['head']['abstracts'] # ABSTRACT
# json_all_data_Snowball_1_citations[0]['abstracts-retrieval-response']['item']['bibrecord']['item-info']['itemidlist']['ce:doi'] #  DOI

In [ ]:
# You now have to save it to a separate folder where names of the files don't repeat and are all of the same format
# CHANGE THE NUMBERING OF NAME FILES

output_dir = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info"

for i in range(0, len(json_all_data_Snowball_1_citations)):
    citation = json_all_data_Snowball_1_citations[i]
    output_path = os.path.join(output_dir, f'single_paper_abstract_info_{i}.json') # LATER ON CHANGE to  i + LAST EXISTING NUMBER
    with open(output_path, 'w') as f:
        json.dump(citation, f, indent=2)
    print(f"Saved data to {output_path}")

In [ ]:
try:
    with open('Snowball+2_citations_Single_paper_info.pkl', 'rb') as f:
        json_citations_Snowball_2 = pickle.load(f)
    print("Successfully loaded 'Snowball+2_citations_Single_paper_info.pkl'")
except FileNotFoundError:
    print("Error: 'Snowball+2_citations_Single_paper_info.pkl' not found.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

In [ ]:
json_citations_Snowball_2[0]['abstracts-retrieval-response']['item']['bibrecord']['tail']['bibliography']['reference'][0]['ref-fulltext']

"Agarwal, B., A Field of One's Own: Gender and Land Rights in South Asia. 1994, Cambridge University Press."

In [ ]:
json_citations_Snowball_2[0]['abstracts-retrieval-response']['item']['bibrecord']['tail']['bibliography']['reference']

In [ ]:
# Create an empty list to hold the table data
table_data = []

# Iterate through each item in json_all_data_Snowball_1_citations
for item_data in json_citations_Snowball_2:
    try:
        # Extract the required fields from the current item
        citation_title = item_data['abstracts-retrieval-response']['item']['bibrecord']['head']['citation-title']
        doi = item_data['abstracts-retrieval-response']['item']['bibrecord']['item-info']['itemidlist']['ce:doi']
        references = item_data['abstracts-retrieval-response']['item']['bibrecord']['tail']['bibliography']['reference']

        # Iterate through the references for the current item
        for ref in references:
            # Check if the reference is a dictionary before trying to access keys
            if isinstance(ref, dict):
                # Extract the full reference text, handling cases where it might be missing
                ref_fulltext = ref.get('ref-fulltext', 'N/A') # Use .get() with a default value

                # Append a row to the table data list
                table_data.append([citation_title, doi, ref_fulltext])
            else:
                print(f"Skipping non-dictionary reference: {ref}")

    except (KeyError, TypeError) as e:
        # Handle cases where the expected keys might be missing in an item
        print(f"Could not process item due to missing key or incorrect structure: {e}")
        continue # Skip to the next item if an error occurs

# Create a Pandas DataFrame from the collected data
df_references = pd.DataFrame(table_data, columns=['Citation Title', 'DOI', 'Reference Fulltext'])

# Display the resulting table
print(df_references.head()) # Print the first few rows as a sample

In [ ]:
# Keep only uniques
df_references = df_references.drop_duplicates('Reference Fulltext', keep=False)

In [ ]:
###### Continue HERE!!!!!!!!
# Split df_references to three chunks, and then find DOI's for all of them

,Citation Title,DOI,Reference Fulltext
0,"“The family farms together, the decisions, how...",10.1016/j.landusepol.2017.10.048,"Agarwal, B., A Field of One's Own: Gender and ..."
1,"“The family farms together, the decisions, how...",10.1016/j.landusepol.2017.10.048,"Agarwal, B., Bargaining and gender relations: ..."
2,"“The family farms together, the decisions, how...",10.1016/j.landusepol.2017.10.048,"Agarwal, B., Gender and land rights revisited:..."
3,"“The family farms together, the decisions, how...",10.1016/j.landusepol.2017.10.048,"Agarwal, B., Women's land rights and the trap ..."
4,"“The family farms together, the decisions, how...",10.1016/j.landusepol.2017.10.048,"Alden Wily, L., ‘The law is to blame’: the vul..."
...,...,...,...
2692889,Environmental attitudes in cross-national pers...,10.1093/esr/jcp018,"Stern, P. C. and Dietz, T. (1994). The Value B..."
2692890,Environmental attitudes in cross-national pers...,10.1093/esr/jcp018,"Van Liere, K. and Dunlap, R. E. (1980). The So..."
2692891,Environmental attitudes in cross-national pers...,10.1093/esr/jcp018,"Wilson, M. et al. (1996). Sex Differences in V..."
2692892,Environmental attitudes in cross-national pers...,10.1093/esr/jcp018,"York, R., Rosa, E. A. and Dietz, T. (2003). Fo..."


In [ ]:
df_references.to_csv('df_references_citations_Snowball_2.csv')

In [ ]:
df_references = pd.read_csv('edgelist_snowball_1_citations.csv') # df_references.csv is edgelist_snowball_1_citations
df_references = df_references.drop(columns=['Unnamed: 0'])

In [ ]:
# Get DOI from a reference
cr = Crossref(timeout=60)  # Increase the timeout to 60 seconds

chunk_size = 1000
for start_index in range(0, len(df_references), chunk_size):
    end_index = min(start_index + chunk_size, len(df_references))
    for i in range(start_index, end_index):
        # Check if the DOI column is NaN or None for the current row
        if pd.isna(df_references.iloc[i, 3]):
            try:
                # Add a small delay to avoid overwhelming the API
                time.sleep(0.5)  # Wait for 0.5 seconds between requests
                result = cr.works(query = df_references.iloc[i,2])
                if 'message' in result and 'items' in result['message'] and result['message']['items']:
                    DOI_TO_SEARCH = result['message']['items'][0]['DOI']
                    df_references.iloc[i,3] = DOI_TO_SEARCH
                else:
                    # Handle cases where the query doesn't return expected structure or items
                    print(f"No results found for query: {df_references.iloc[i,2]}")
                    continue
            except requests.exceptions.Timeout:
                print(f"Request timed out for query: {df_references.iloc[i,2]}")
                # You might want to implement a retry mechanism here
                continue
            except Exception as e:
                print(f"An error occurred at index {i}: {e}")
                continue
    # Save the DataFrame periodically
    df_references.to_csv('df_references.csv')


In [ ]:
df_references.to_csv('/content/df_references.csv')

In [ ]:
# remove duplicate rows from df_references
df_references = pd.read_csv('edgelist_snowball_1_citations.csv')
df_references = df_references.drop(columns=['Unnamed: 0'])
df_references = df_references.drop_duplicates()
# get references for + 2 citations
DOI_for_citarions_plus_1 = df_references[['Reference_DOI']]
DOI_for_citarions_plus_1 = DOI_for_citarions_plus_1.drop_duplicates()
# drop rows from DOI_for_citarions_plus_2 that are Reference_DOI is NA
DOI_for_citarions_plus_1 = DOI_for_citarions_plus_1.dropna(subset=['Reference_DOI'])

In [ ]:
DOI_for_citarions_plus_1.to_csv('DOI_for_citarions_plus_1.csv')

,Reference_DOI
0,10.3386/w8456
1,10.1080/08941920701537007
2,10.1002/wrcr.20228
3,10.2737/rmrs-rp-71
4,10.2737/int-gtr-184
...,...
137560,10.1016/j.foreco.2005.12.037
137561,10.18356/9789211561418c057
137564,10.4324/9780203403419-39
137565,10.7146/ln.v0i2.19089


In [ ]:
scopus_urls = get_scopus_urls_ABSTRACT(DOI_for_citarions_plus_2['Reference_DOI'], api_key)
scopus_urls_df = pd.DataFrame({'scopus_urls': scopus_urls})
scopus_urls_df.to_csv('scopus_urls_df_for_snowball_1_citations.csv', index = False)


In [ ]:
%pwd

'/content/drive/MyDrive/Colab Notebooks/Forest_policy_review'

In [ ]:
# copy the content from  /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info_download to /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info. If there are two files with the same name, keep them both and rename one

import shutil

source_directory = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info_download"
destination_directory = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Single_paper_info"

# Create the destination directory if it doesn't exist
os.makedirs(destination_directory, exist_ok=True)

# List files in the source directory
files_to_copy = os.listdir(source_directory)

for filename in files_to_copy:
    source_path = os.path.join(source_directory, filename)
    destination_path = os.path.join(destination_directory, filename)

    # Check if the destination file already exists
    if os.path.exists(destination_path):
        # File with the same name exists, find a new name
        base, extension = os.path.splitext(filename)
        counter = 1
        new_filename = f"{base}_{counter}{extension}"
        new_destination_path = os.path.join(destination_directory, new_filename)

        while os.path.exists(new_destination_path):
            counter += 1
            new_filename = f"{base}_{counter}{extension}"
            new_destination_path = os.path.join(destination_directory, new_filename)

        print(f"File '{filename}' already exists. Copying as '{new_filename}'.")
        shutil.copy2(source_path, new_destination_path) # Use copy2 to preserve metadata
    else:
        # File does not exist, just copy it
        print(f"Copying '{filename}' to '{destination_directory}'.")
        shutil.copy2(source_path, destination_path)

In [ ]:
# THIS WORKS - GET METADATA FOR A SINGLE PAPER
# https://api.elsevier.com/content/abstract/eid/2-s2.0-71749119078?apiKey=XYZ

url = "https://api.elsevier.com/content/abstract/eid/2-s2.0-71749119078?apiKey=XYZ&httpAccept=application/json"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    print(json.dumps(data, indent=2))
else:
    print(f"Error: Unable to fetch data. Status code: {response.status_code}")
response.text

In [ ]:
####

# Making a single file listing ID info on all papers in -2 to +2 range

In [ ]:
# At the beggining FPE core does not have abstracts!
# Get these from + network info

In [ ]:
FPE_core = pd.read_csv('FPE_core.csv')

In [ ]:
basic_paper_data = pd.DataFrame()

In [ ]:
FPE_core

In [ ]:
basic_paper_data['EID'] = FPE_core['EID']
basic_paper_data['DOI'] = FPE_core['DOI']
basic_paper_data['Title'] = FPE_core['Title']
basic_paper_data['Abstract'] = FPE_core['Abstract']

In [ ]:
# Do this twice, for -1 and -2 snowball (cited by)
with open('json_cited_by_Snowball_1.pkl', 'rb') as f:
    json_cited_by_Snowball_1 = pickle.load(f)
basic_paper_data_2 = pd.DataFrame(columns = ['EID', 'DOI', 'Title', 'Abstract'])



for i in range(0, len(json_cited_by_Snowball_1)):
    search_EID = json_cited_by_Snowball_1[i]['search-results']['opensearch:Query'][
        '@searchTerms']  # searchTerms
    search_EID = search_EID.replace("REFEID(", "").replace(")", "")
    search_doi = None

    for j in range(0, len(json_cited_by_Snowball_1[i]['search-results']['entry'])):
        # Check if 'eid' key exists before accessing it
        entry = json_cited_by_Snowball_1[i]['search-results']['entry'][j]
        if 'eid' in entry:
            one_eid = entry['eid']
        else:
            # Handle the case where 'eid' is missing, e.g., skip this entry or assign a default value
            print(f"Warning: 'eid' key missing in entry {j} of search results {i}. Skipping this entry.")
            continue  # Skip to the next entry

        # Check if 'prism:doi' key exists before accessing it
        one_doi = entry.get('prism:doi')
        # If 'prism:doi' is not found, assign None or an empty string
        if one_doi is None:
            one_doi = ''  # or one_doi = None
        # Check if 'dc:title' key exists before accessing it
        one_title = entry.get('dc:title', '') # Assign empty string if not found
        # Check if 'dc:description' key exists before accessing it
        one_abstract = entry.get('dc:description', '')  # Assign empty string if not found
        one_paper_data = pd.DataFrame({
            'EID': [one_eid],
            'DOI': [one_doi],
            'Title': [one_title],
            'Abstract': [one_abstract]
        })
        if one_doi not in basic_paper_data_2['DOI'].values:  # add one_paper_data to basic_paper_data only if one_doi is not in basic_paper_data['DOI']
            basic_paper_data_2 = pd.concat([basic_paper_data_2, one_paper_data],
                                         ignore_index=True)

In [ ]:
basic_paper_data = pd.concat([basic_paper_data, basic_paper_data_2], ignore_index=True)

In [ ]:
basic_paper_data.to_csv('basic_paper_data_minus_one_minus_two.csv')

In [ ]:
# These are papers from - / minus networks. You should have information on + for all of this papers as well (i.e. whom have they cited)
# Get this information by subsetting papers that you already have in +1 and +2, and then downlad this missing paper info

In [ ]:
basic_paper_data_minus_one_minus_two = pd.read_csv('basic_paper_data_minus_one_minus_two.csv')
basic_paper_data_minus_one_minus_two = basic_paper_data_minus_one_minus_two.drop(columns=['Unnamed: 0'])
basic_paper_data_minus_one_minus_two = basic_paper_data_minus_one_minus_two.drop_duplicates()

/tmp/ipython-input-2452676686.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  basic_paper_data_minus_one_minus_two = pd.read_csv('basic_paper_data_minus_one_minus_two.csv')


In [ ]:
# Get basic data from the forward networks

with open('Snowball+1_citations_Single_paper_info.pkl', 'rb') as f:
    Snowball_1_citations_Single_paper_info = pickle.load(f)
basic_paper_data_3 = pd.DataFrame(columns = ['EID', 'DOI', 'Title', 'Abstract'])
for i in range(len(Snowball_1_citations_Single_paper_info)):
    search_EID = Snowball_1_citations_Single_paper_info[i]['abstracts-retrieval-response']['coredata']['eid']
    search_doi = Snowball_1_citations_Single_paper_info[i]['abstracts-retrieval-response']['coredata']['prism:doi']
    search_title = Snowball_1_citations_Single_paper_info[i]['abstracts-retrieval-response']['coredata']['dc:title']
    search_abstract = Snowball_1_citations_Single_paper_info[i]['abstracts-retrieval-response']['coredata'].get('dc:description', '') # Use .get() with a default value
    one_paper_data = pd.DataFrame({'EID': [search_EID], 'DOI': [search_doi], 'Title': [search_title], 'Abstract': [search_abstract]})
    basic_paper_data_3 = pd.concat([basic_paper_data_3, one_paper_data], ignore_index=True)
basic_paper_data_3 = basic_paper_data_3.drop_duplicates()

with open('Snowball+2_citations_Single_paper_info.pkl', 'rb') as f:
    Snowball_2_citations_Single_paper_info = pickle.load(f)

basic_paper_data_4 = pd.DataFrame(columns = ['EID', 'DOI', 'Title', 'Abstract'])
# Check if Snowball_2_citations_Single_paper_info is not None before iterating
if Snowball_2_citations_Single_paper_info is not None:
    for i in range(len(Snowball_2_citations_Single_paper_info)):
        # Check if the current item is not None before processing
        if Snowball_2_citations_Single_paper_info[i] is not None and \
           'abstracts-retrieval-response' in Snowball_2_citations_Single_paper_info[i] and \
           Snowball_2_citations_Single_paper_info[i]['abstracts-retrieval-response'] is not None and \
           'coredata' in Snowball_2_citations_Single_paper_info[i]['abstracts-retrieval-response'] and \
           Snowball_2_citations_Single_paper_info[i]['abstracts-retrieval-response']['coredata'] is not None:

            coredata = Snowball_2_citations_Single_paper_info[i]['abstracts-retrieval-response']['coredata']
            search_EID = coredata.get('eid')
            search_doi = coredata.get('prism:doi')
            search_title = coredata.get('dc:title')
            search_abstract = coredata.get('dc:description', '')

            # Only create a DataFrame and concatenate if essential data is present
            if search_EID and search_title:
                one_paper_data = pd.DataFrame({'EID': [search_EID], 'DOI': [search_doi], 'Title': [search_title], 'Abstract': [search_abstract]})
                basic_paper_data_4 = pd.concat([basic_paper_data_4, one_paper_data], ignore_index=True)
        else:
            print(f"Warning: Skipping entry at index {i} due to missing or None data.")
else:
    print("Warning: Snowball_1_citations_Single_paper_info is None.")

basic_paper_data_plus_one_plus_two = pd.concat([basic_paper_data_3, basic_paper_data_4], ignore_index=True)

basic_paper_data_plus_one_plus_two = basic_paper_data_plus_one_plus_two.drop_duplicates()


In [ ]:
# Now identify the papers that don't have forward / basic paper data info, which are just identified by the minus networks
merged_df = basic_paper_data_minus_one_minus_two.merge(basic_paper_data_plus_one_plus_two[['EID', 'DOI']], on=['EID', 'DOI'], how='left', indicator=True)
new_df = merged_df[merged_df['_merge'] == 'left_only'].drop(columns=['_merge'])
# Then download data for these papers, i.e. their basic info and whom have they cited.
# Split it to under 100k chunks - and download this data
df_new_1, df_new_2, df_new_3 = np.array_split(new_df, 3)
scopus_urls_1 = get_scopus_urls_ABSTRACT(df_new_1['DOI'], api_key)
scopus_urls_2 = get_scopus_urls_ABSTRACT(df_new_2['DOI'], api_key)
scopus_urls_3 = get_scopus_urls_ABSTRACT(df_new_3['DOI'], api_key)
scopus_urls_1_df = pd.DataFrame({'scopus_urls': scopus_urls_1})
scopus_urls_2_df = pd.DataFrame({'scopus_urls': scopus_urls_2})
scopus_urls_3_df = pd.DataFrame({'scopus_urls': scopus_urls_3})
scopus_urls_1_df.to_csv('Single_paper_info_citations_from_minus_1.txt', sep='\t', index=False, header=False)
scopus_urls_2_df.to_csv('Single_paper_info_citations_from_minus_2.txt', sep='\t', index=False, header=False)
scopus_urls_3_df.to_csv('Single_paper_info_citations_from_minus_3.txt', sep='\t', index=False, header=False)

# end of chunk on combining lists of 'cited by'

In [ ]:
file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/Snowball+1_cited_by"
files = list_files(file_path)

# Initialize json_all_data as a list to store the combined data
json_all_data_Snowball_1 = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data_Snowball_1.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

# Chunk on initial communities - to cut those that are really out of FPE

In [ ]:
# in edgelist_0, check NaN by column. There are none in EIDs, so we'll proceed based on these - and not DOI
edgelist_0.isna().sum()
edgelist_all = edgelist_0[['source_EID','target_EID']] # save a clean edgelist that Networkx can use
edgelist_all['weight'] = 1 # add weight

# Community detection
Row_list =[]
# Iterate over each row
for index, rows in edgelist_all.iterrows():
    # Create list for the current row
    my_list =[rows.source_EID, rows.target_EID, rows.weight]
    # append the list to the final list
    Row_list.append(my_list)
graph = nx.DiGraph()
graph.add_weighted_edges_from(Row_list)
community = nx.community.louvain_communities(graph, seed=123)
community = [list(sub_community) for sub_community in community]


# Make a file that lists EIDs of cummunity members as a row
df_communities_all = pd.DataFrame(columns=['index', 'size', 'EIDs'])
for i in range(0, len(community)):
    # community is a list of lists, so you can access the inner list directly
    EIDs = list(community[i])
    df_communities = pd.DataFrame(columns=['index', 'size', 'EIDs'])
    # Convert the list of DOIs to a comma-separated string
    EIDs_str = ', '.join(EIDs)
    df_communities.loc[0] = [i, len(EIDs), EIDs_str]
    df_communities_all = pd.concat([df_communities_all, df_communities], ignore_index=True)
# Now you can apply the str.replace() methods
df_communities_all['EIDs'] = df_communities_all['EIDs'].str.replace("'", '')
df_communities_all['EIDs'] = df_communities_all['EIDs'].str.replace('{', '')
df_communities_all['EIDs'] = df_communities_all['EIDs'].str.replace('}', '')


In [ ]:
# Add DOIs, titles and abstracts to df_communities_all based on EID
dois = []
titles = []
abstracts = []

# Iterate through each row in df_communities_all
for index, row in df_communities_all.iterrows():
    # Split the EIDs string into a list of individual EIDs
    eid_list = [eid.strip() for eid in row['EIDs'].split(',')]

    # Initialize lists to store DOI, title, and abstract for the current community
    community_dois = []
    community_titles = []
    community_abstracts = []

    # Iterate through each EID in the current community
    for eid in eid_list:
        # Find the corresponding entry in basic_paper_data
        matching_rows = basic_paper_data[basic_paper_data['EID'] == eid]

        # If a match is found
        if not matching_rows.empty:
            # Append the DOI, title, and abstract to the community lists
            community_dois.append(matching_rows['DOI'].iloc[0])
            community_titles.append(matching_rows['Title'].iloc[0])
            community_abstracts.append(matching_rows['Abstract'].iloc[0])
        else:
            # Handle the case where no match is found (e.g., append empty strings or NaN)
            community_dois.append('')
            community_titles.append('')
            community_abstracts.append('')

    # Join the lists for the current community into comma-separated strings
    dois.append(', '.join(community_dois))
    titles.append(', '.join(community_titles))
    abstracts.append(', '.join(community_abstracts))

# Add the new columns to df_communities_all
df_communities_all['DOIs'] = dois
df_communities_all['Titles'] = titles
df_communities_all['Abstracts'] = abstracts


In [ ]:
# Check if the transformation of data from above is correct - compare tedgelist_0 to df_communities_all
target_eid = '2-s2.0-85219699174'
result = edgelist_0[(edgelist_0['source_EID'] == target_eid) | (edgelist_0['target_EID'] == target_eid)]

In [ ]:
# Here add cosine similariy columns - based on titles and abstracts from FPE

FPE_core = pd.read_csv('FPE_core.csv')
basic_paper_data = FPE_core[['EID', 'DOI', 'Title', 'Abstract']]
FPE_core_titles = basic_paper_data['Title']
FPE_core_titles = ", ".join(FPE_core_titles.astype(str))
FPE_core_abstracts = basic_paper_data['Abstract']
FPE_core_abstracts = ", ".join(FPE_core_abstracts.astype(str))
# remove stopwords from FPE_core_titles and FPE_core_abstracts
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
words = FPE_core_titles.lower().split()
filtered_words = [word for word in words if word not in stop_words]
FPE_core_titles_no_stopwords = " ".join(filtered_words)
words = FPE_core_abstracts.lower().split()
filtered_words = [word for word in words if word not in stop_words]
FPE_core_abstracts_no_stopwords = " ".join(filtered_words)

total_start = time.time()

similarity_cosine = []
for i in range(0,df_communities_all.shape[0]):
    text_to_compare = df_communities_all['Titles'][i]
    texts = [FPE_core_titles_no_stopwords, text_to_compare]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)
    similarity = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
    similarity_cosine.append(similarity)

similarity_cosine = pd.Series(similarity_cosine)
df_communities_all['similarity_title_cosine'] = similarity_cosine

similarity_cosine = []
for i in range(0,df_communities_all.shape[0]):
    text_to_compare = df_communities_all['Abstracts'][i]
    texts = [FPE_core_abstracts_no_stopwords, text_to_compare]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)
    similarity = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
    similarity_cosine.append(similarity)

similarity_cosine = pd.Series(similarity_cosine)
df_communities_all['similarity_abstract_cosine'] = similarity_cosine

total_end = time.time()
total_elapsed_seconds = total_end - total_start
total_elapsed_time = format_timespan(total_elapsed_seconds)
send_push_final(total_elapsed_time)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Networkx

# Saving and loading list https://stackoverflow.com/questions/27745500/how-to-save-a-list-to-a-file-and-read-it-as-a-list-type
community = [list(sub_community) for sub_community in community]
with open("community.json", 'w') as f:
    json.dump(community, f, indent=2)

# Chunk on communities to cut ends

# CHUNK ON LLM - OLLAMA

In [ ]:
df_communities_all = pd.read_csv('df_communities_all.csv')
FPE_core = pd.read_csv('FPE_core.csv')
FPE_basic_paper_data = FPE_core[['EID', 'DOI', 'Title', 'Abstract']]

In [ ]:
# https://medium.com/@abonia/running-ollama-in-google-colab-free-tier-545609258453
!curl https://ollama.ai/install.sh | sh
# ollama serve & # Run this in console
# !ollama pull deepseek-r1:70b

In [ ]:
!ollama pull deepseek-r1:70b

In [ ]:
!pip install ollama

In [ ]:
import ollama
response = ollama.chat(
    model="deepseek-r1:70b",
    messages=[
        {"role": "user", "content": "24 plus 24 is?"},
    ],
)
print(response["message"]["content"])

<think>

</think>

24 plus 24 equals **48**.


# CUSTOM TRAINING LLM HUGGINGFACE

In [ ]:
# https://medium.com/@tossy21/fine-tuning-modernbert-on-imdb-movie-review-comments-for-text-classification-0df8ace77259
!pip install git+https://github.com/huggingface/transformers.git
!pip install datasets
!pip install dill==0.3.5.1

In [ ]:
from transformers import ModernBertForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import random

In [ ]:
model = ModernBertForSequenceClassification.from_pretrained('answerdotai/ModernBERT-base')
tokenizer = AutoTokenizer.from_pretrained('answerdotai/ModernBERT-base')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
imdb_data = load_dataset('imdb')

In [ ]:
# Load the data
FPE_core = pd.read_csv('FPE_core.csv')
FPE_basic_paper_data = FPE_core[['EID', 'DOI', 'Title', 'Abstract']]
df_communities_all = pd.read_csv('df_communities_all.csv')
basic_paper_data = pd.read_csv('basic_paper_data.csv')
# format FPE abstracts to list and add a label 0 that it's in the sample
FPE_formatted_data = []
for abstract in FPE_basic_paper_data['Abstract']:
    FPE_formatted_data.append({'text': str(abstract), 'label': 0}) # Changed abstract to str(abstract)

# from df_communities_all, extract individual values in column EIDs from rows where target_class_abstract is 1
eids_list = []
for index, row in df_communities_all.iterrows():
    if row['target_class_abstract'] == 1:
      eids_str = row['EIDs']
      eids = [eid.strip() for eid in eids_str.split(',')]
      eids_list.extend(eids)
eids_list
# Make the same list format for data that is not in FPE
Not_FPE_formatted_data = []
for eid in eids_list:
    matching_rows = basic_paper_data[basic_paper_data['EID'] == eid]
    if not matching_rows.empty:
        abstract = matching_rows['Abstract'].iloc[0]
        Not_FPE_formatted_data.append({'text': str(abstract), 'label': 1}) # label: 0 and changed abstract to str(abstract) # label: -1
    else:
        print(f"Warning: EID '{eid}' not found in basic_paper_data")

# randomly select 250 elements of FPE_formatted_data
import random
FPE_formatted_data_250 = random.sample(FPE_formatted_data, 250)
all_data = FPE_formatted_data_250 + Not_FPE_formatted_data # join it
random.shuffle(all_data)  # Shuffle the data randomly
# split all_data randomly to train_dataset and test_dataset, where 80% goes to train_dataset
train_size = int(0.8 * len(all_data))
train_dataset = all_data[:train_size]
test_dataset = all_data[train_size:]

# Convert lists to Dataset objects using the 'datasets' library
from datasets import Dataset
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True)



train_dataset = train_dataset.map(tokenize, batched=True, batch_size=4)
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=4)
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map:   0%|          | 0/391 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

In [ ]:
# prompt: make a trainer and train on train_dataset and  test_dataset

import torch
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import random
from transformers import AutoTokenizer, ModernBertForSequenceClassification # Assuming you are using ModernBert
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords

# Assuming model and tokenizer are already defined and loaded
# model = ModernBertForSequenceClassification.from_pretrained('answerdotai/ModernBERT-base')
# tokenizer = AutoTokenizer.from_pretrained('answerdotai/ModernBERT-base')

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train the model
trainer.train()


<ipython-input-4-b978f3710a26>:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


TrainOutput(global_step=98, training_loss=0.6133459830770687, metrics={'train_runtime': 16.4487, 'train_samples_per_second': 23.771, 'train_steps_per_second': 5.958, 'total_flos': 93133378789248.0, 'train_loss': 0.6133459830770687, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.3536865711212158,
 'eval_accuracy': 0.8163265306122449,
 'eval_f1': 0.7857142857142857,
 'eval_precision': 0.9166666666666666,
 'eval_recall': 0.6875,
 'eval_runtime': 1.5708,
 'eval_samples_per_second': 62.389,
 'eval_steps_per_second': 15.916,
 'epoch': 1.0}

In [ ]:
# prompt: use trainer to predict labels for test_dataset['text']

import numpy as np
from transformers import Trainer

# Assuming 'trainer' and 'test_dataset' are already defined and loaded as in your provided code.

predictions = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Now 'predicted_labels' contains the predicted labels for test_dataset['text']
predicted_labels


array([1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 1])

In [ ]:
def labelcheck(dataset):
    labels = dataset['label']
    # convert labels tensor to numpy array
    labels_np = labels.cpu().numpy()

    count_0 = np.count_nonzero(labels_np == 0)
    count_1 = np.count_nonzero(labels_np == 1)

    print(f"Total count of label 0: {count_0}")
    print(f"Total count of label 1: {count_1}")

labelcheck(train_dataset)
labelcheck(test_dataset)

Total count of label 0: 208
Total count of label 1: 183
Total count of label 0: 42
Total count of label 1: 56


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 9.488086050168931e-08,
 'eval_accuracy': 1.0,
 'eval_f1': 0.0,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_runtime': 4.0696,
 'eval_samples_per_second': 24.081,
 'eval_steps_per_second': 6.143,
 'epoch': 1.0}

In [ ]:
# https://medium.com/@jiaqixu012/run-deepseek-r1-on-local-machine-ollama-python-vs-code-f-f31d95ecf6ac
# https://abhishek-maheshwarappa.medium.com/fine-tuning-deepseek-llm-adapting-open-source-ai-for-your-needs-12a7e5572fa5
# https://medium.com/@tossy21/fine-tuning-modernbert-on-imdb-movie-review-comments-for-text-classification-0df8ace77259
!pip install -U torch datasets accelerate peft bitsandbytes
!pip uninstall torchvision -y
!pip install torchvision --no-cache-dir
# !pip install transformers==4.31.0 # Install the specific version of transformers to resolve the conflict

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, TrainerCallback
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import os
from google.colab import userdata

# Check if CUDA is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA device:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("CUDA is not available, using CPU.")


model_name = "deepseek-ai/deepseek-llm-7b-base" # 1.5B, 7B, 8B, 14B, 32B, and 70B
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
)

# Securely load Hugging Face token
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ deepseek-r1:70b LLM loaded")

# Load and process dataset (example using IMDB dataset)
dataset = load_dataset("imdb")

def tokenize_function(examples):
    inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(100))


training_args = TrainingArguments(
    output_dir="./results",
    eval_steps=500,  # Evaluate every 500 steps. You can adjust based on your needs
    learning_rate=3e-4,  # Lower learning rate for LoRA fine-tuning
    per_device_train_batch_size=1,  # Reduce batch size for memory efficiency
    gradient_accumulation_steps=8,  # Simulate larger batch size
    num_train_epochs=0.5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,  # Mixed precision training,
    report_to="none"  # Disable WandB integration
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset
)

print("🚀 Trainer Initialized!")
trainer.train()
print("Training complete!")

In [ ]:
def tokenize_function(examples):
    inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)
# Subset the dataset for faster experimentation
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(100))
# Print a sample tokenized entry
print("Tokenized Sample:")
print(small_train_dataset[0])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenized Sample:
{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1, 'input_ids': [100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001, 100001,

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch",
    learning_rate=3e-4,  # Lower learning rate for LoRA fine-tuning
    per_device_train_batch_size=1,  # Reduce batch size for memory efficiency
    gradient_accumulation_steps=8,  # Simulate larger batch size
    num_train_epochs=0.5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,  # Mixed precision training
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
)
print("🚀 Trainer Initialized!")

print("🚀 Starting Fine-Tuning...")
trainer.train()

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


🚀 Trainer Initialized!
🚀 Starting Fine-Tuning...


Step,Training Loss


TrainOutput(global_step=31, training_loss=1.4495810231854838, metrics={'train_runtime': 46.0687, 'train_samples_per_second': 5.427, 'train_steps_per_second': 0.673, 'total_flos': 4948153740558336.0, 'train_loss': 1.4495810231854838, 'epoch': 0.496})

In [ ]:
eval_results = trainer.evaluate()
eval_results

{'eval_runtime': 4.4018,
 'eval_samples_per_second': 22.718,
 'eval_steps_per_second': 2.953,
 'epoch': 0.496}

In [ ]:
# prompt: use trainer to predict labels on test_dataset

predictions = trainer.predict(test_dataset['text'])
predictions


In [ ]:
#### HELP ##### SKIP
df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[0,5] # titles
df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[0,6] # abstracts
df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[0,7] # short title LLM 5 words
df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[0,8] # short title LLM 15 words
df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[0,9] # short abstract 300 words
"content": f"summarize the following titles: {df_communities_all_sub_sub.iloc[i,5]} into a single new. This new title should be up to 5 words. Respond just with this new summary title"
"content": f"summarize the following text: {df_communities_all_sub_sub.iloc[0,6]}. The summary should be up to 300 words. Respond just with this summary of all providedresearch papers"

In [ ]:
# Get last index where you have data
last_string_index = -1
for i in range(len(df_communities_all_sub_sub_tiles_abstracts_summaries['abstract_LLM']) - 1, -1, -1):
    if isinstance(df_communities_all_sub_sub_tiles_abstracts_summaries['abstract_LLM'].iloc[i], str):
        last_string_index = i
        break
last_string_index

618

In [ ]:
total_start = time.time()

for i in range(last_string_index, len(df_communities_all_sub_sub_tiles_abstracts_summaries)): # last_string_index
    response = ollama.chat(
        model="deepseek-r1:70b",
        messages=[
            {"role": "user", "content": f"summarize the following text: {df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[i,6]}. The summary should be up to 300 words. Respond just with this summary of all providedresearch papers"},
        ],
    )
    response_text = response["message"]["content"]
    response_text = response_text.split('\n\n')
    merged_text = "".join(response_text[-3:])
    df_communities_all_sub_sub_tiles_abstracts_summaries.iloc[i,9] = merged_text # response_text[-1] is for titles
    df_communities_all_sub_sub_tiles_abstracts_summaries.to_csv('df_communities_all_sub_sub_tiles_abstracts_summaries.csv')

total_end = time.time()
total_elapsed_seconds = total_end - total_start
total_elapsed_time = format_timespan(total_elapsed_seconds)
send_push_final(total_elapsed_time)


# LLM CHUNK ENDS

In [ ]:
# Bibliometric data downloaded from SCOPUS from all FPE papers
# bibliographic_data_from_Emil = pd.read_csv('bibliographic_data_from_Emil.csv')
df_papers_for_Carl_Emil = pd.read_csv('df_papers_for_Carl_Emil.csv')
edgelist_all_FPE_papers = pd.read_csv('edgelist_all_FPE_papers.csv')
edgelist_snowball_1 = pd.read_csv('edgelist_snowball_1.csv')
edgelist_snowball_2 = pd.read_csv('edgelist_snowball_2.csv')
edgelist_all_FPE_papers = edgelist_all_FPE_papers.drop(columns=['Unnamed: 0'])
edgelist_snowball_1 = edgelist_snowball_1.drop(columns=['Unnamed: 0'])
edgelist_snowball_2 = edgelist_snowball_2.drop(columns=['Unnamed: 0'])
# edgelist_all = edgelist_all.drop(columns=['Unnamed: 0'])


In [ ]:
edgelist_all_cut = edgelist_all[edgelist_all['target'].isin(df_papers_for_Carl_Emil['DOI'])]
edgelist_all_cut

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,source,target,weight
0,10.1016/j.forpol.2024.103231,10.1016/j.landusepol.2023.106617,1
1,10.1016/j.forpol.2024.103231,10.1038/s43247-023-00771-z,1
2,10.1016/j.forpol.2024.103231,10.1080/08913811.2010.541686,1
3,10.1016/j.forpol.2024.103231,10.1177/0738894210388127,1
5,10.1016/j.forpol.2024.103231,10.1016/j.forpol.2022.102717,1
...,...,...,...
2271560,10.1108/jec-02-2015-0016,10.1525/cmr.2011.53.3.40,1
2271571,10.1108/jec-02-2015-0016,10.1080/01490409209513155,1
2271667,10.1108/mrr-06-2015-0147,10.1177/014920639101700108,1
2271687,10.1108/mrr-06-2015-0147,10.5465/amr.1995.9508080331,1


In [ ]:
unique_source_count = edgelist_all_cut['target'].nunique()
unique_source_count

141961

In [ ]:
# Here you generate edgelist for all papers in FPE Journal, i.e. all papers referenced in the FPE journal
total_start = time.time()

edgelist_all_FPE_papers = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame

for i in range(0,len(FPE_core)):
    result = Crossref().works(query = f'{FPE_core.iloc[i,3]}')
    #Check if the result contains the 'message' and 'items' keys before accessing them
    if 'message' in result and 'items' in result['message'] and len(result['message']['items']) > 0:
        result = result['message']['items'][0]
        edgelist_single_paper = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame
        references_doi = []
        references_title = []
        # Check if the result contains the 'reference' key before accessing it
        if 'reference' in result:
            for j in range(0,len(result['reference'])):
                try:
                    # Ensure both DOI and title are extracted before appending
                    doi = result['reference'][j].get('DOI')
                    title = result['reference'][j].get('article-title')
                    if doi and title: # Append only if both values are present
                        references_doi.append(doi)
                        references_title.append(title)
                except:
                    continue
        references_df = pd.DataFrame({'source': [result['DOI']] * len(references_doi), 'target': references_doi, 'target_title': references_title})
        edgelist_single_paper = pd.concat([edgelist_single_paper, references_df], ignore_index=True)
        edgelist_all_FPE_papers = pd.concat([edgelist_all_FPE_papers, edgelist_single_paper], ignore_index=True)
    else:
        print(f"Skipping record {i} due to missing data in Crossref response.")

edgelist_all_FPE_papers.to_csv('edgelist_all_FPE_papers.csv')

total_end = time.time()
total_elapsed_seconds = total_end - total_start
total_elapsed_time = format_timespan(total_elapsed_seconds)
send_push_final(total_elapsed_time)

In [ ]:
edgelist_all_FPE_papers = pd.read_csv('edgelist_all_FPE_papers.csv')
edgelist_all_FPE_papers = edgelist_all_FPE_papers[edgelist_all_FPE_papers["target"].str.contains("forpol") == False] # drop those references that are within FPE
unique_references_title = edgelist_all_FPE_papers['target_title'].unique()
df_1, df_2, df_3, df_4 = np.array_split(unique_references_title, 4)

In [ ]:
total_start = time.time()

edgelist_all_FPE_papers = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame

for i in range(0,len(df_1)):
    result = Crossref().works(query = f'{df_1[i]}') ######### MODIFY THIS THIS
    #Check if the result contains the 'message' and 'items' keys before accessing them
    if 'message' in result and 'items' in result['message'] and len(result['message']['items']) > 0:
        result = result['message']['items'][0]
        edgelist_single_paper = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame
        references_doi = []
        references_title = []
        # Check if the result contains the 'reference' key before accessing it
        if 'reference' in result:
            for j in range(0,len(result['reference'])):
                try:
                    # Ensure both DOI and title are extracted before appending
                    doi = result['reference'][j].get('DOI')
                    title = result['reference'][j].get('article-title')
                    if doi and title: # Append only if both values are present
                        references_doi.append(doi)
                        references_title.append(title)
                except:
                    continue
        references_df = pd.DataFrame({'source': [result['DOI']] * len(references_doi), 'target': references_doi, 'target_title': references_title})
        edgelist_single_paper = pd.concat([edgelist_single_paper, references_df], ignore_index=True)
        edgelist_all_FPE_papers = pd.concat([edgelist_all_FPE_papers, edgelist_single_paper], ignore_index=True)
    else:
        print(f"Skipping record {i} due to missing data in Crossref response.")

edgelist_all_FPE_papers.to_csv('edgelist_snowball_1_1.csv')

total_end = time.time()
total_elapsed_seconds = total_end - total_start
total_elapsed_time = format_timespan(total_elapsed_seconds)
send_push_final(total_elapsed_time)

In [ ]:
# Now load the 4 files of snowball +1, merge them, list only ones not previously listed, and remove the duplicates
edgelist_snowball_1_1 = pd.read_csv('edgelist_snowball_1_1.csv')
edgelist_snowball_1_2 = pd.read_csv('edgelist_snowball_1_2.csv')
edgelist_snowball_1_3= pd.read_csv('edgelist_snowball_1_3.csv')
edgelist_snowball_1_4 = pd.read_csv('edgelist_snowball_1_4.csv')
del edgelist_snowball_1_1['Unnamed: 0']
del edgelist_snowball_1_2['Unnamed: 0']
del edgelist_snowball_1_3['Unnamed: 0']
del edgelist_snowball_1_4['Unnamed: 0']
edgelist_snowball_1 = pd.concat([edgelist_snowball_1_1, edgelist_snowball_1_2, edgelist_snowball_1_3, edgelist_snowball_1_4], ignore_index=True,axis=0)
edgelist_snowball_1.to_csv('edgelist_snowball_1.csv') # Save it to file



/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
edgelist_snowball_1 = pd.read_csv('edgelist_snowball_1.csv')
del edgelist_snowball_1['Unnamed: 0']
edgelist_snowball_1 = edgelist_snowball_1[edgelist_snowball_1["target"].str.contains("forpol") == False] # drop those references that are within FPE
unique_references_title = edgelist_snowball_1['target_title'].unique()
df_1, df_2, df_3, df_4 = np.array_split(unique_references_title, 4)

In [ ]:
df_4_1, df_4_2, df_4_3, df_4_4, df_4_5 = np.array_split(df_4, 5)

In [ ]:
total_start = time.time()

edgelist_all_FPE_papers = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame

for i in range(0,len(df_4_1)):
    try:
        result = Crossref().works(query = f'{df_4_1[i]}') ######### MODIFY THIS THIS
        time.sleep(0.025)
        #Check if the result contains the 'message' and 'items' keys before accessing them
        if 'message' in result and 'items' in result['message'] and len(result['message']['items']) > 0:
            result = result['message']['items'][0]
            edgelist_single_paper = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame
            references_doi = []
            references_title = []
            # Check if the result contains the 'reference' key before accessing it
            if 'reference' in result:
                for j in range(0,len(result['reference'])):
                    try:
                        # Ensure both DOI and title are extracted before appending
                        doi = result['reference'][j].get('DOI')
                        title = result['reference'][j].get('article-title')
                        if doi and title: # Append only if both values are present
                            references_doi.append(doi)
                            references_title.append(title)
                    except:
                        continue
            references_df = pd.DataFrame({'source': [result['DOI']] * len(references_doi), 'target': references_doi, 'target_title': references_title})
            edgelist_single_paper = pd.concat([edgelist_single_paper, references_df], ignore_index=True)
            edgelist_all_FPE_papers = pd.concat([edgelist_all_FPE_papers, edgelist_single_paper], ignore_index=True)
        else:
            print(f"Skipping record {i} due to missing data in Crossref response.")
    except Exception:
        pass  # This was not indented properly

edgelist_all_FPE_papers.to_csv('edgelist_snowball_2_4_1.csv')

total_end = time.time()
total_elapsed_seconds = total_end - total_start
total_elapsed_time = format_timespan(total_elapsed_seconds)
send_push_final(total_elapsed_time)

In [ ]:
edgelist_snowball_2_1 = pd.read_csv('edgelist_snowball_2_1.csv')
edgelist_snowball_2_2 = pd.read_csv('edgelist_snowball_2_2.csv')
edgelist_snowball_2_3= pd.read_csv('edgelist_snowball_2_3.csv')
edgelist_snowball_2_4_1 = pd.read_csv('edgelist_snowball_2_4_1.csv')
edgelist_snowball_2_4_2 = pd.read_csv('edgelist_snowball_2_4_2.csv')
edgelist_snowball_2_4_3 = pd.read_csv('edgelist_snowball_2_4_3.csv')
edgelist_snowball_2_4_4 = pd.read_csv('edgelist_snowball_2_4_4.csv')
edgelist_snowball_2_4_5 = pd.read_csv('edgelist_snowball_2_4_5.csv')

del edgelist_snowball_2_1['Unnamed: 0']
del edgelist_snowball_2_2['Unnamed: 0']
del edgelist_snowball_2_3['Unnamed: 0']
del edgelist_snowball_2_4_1['Unnamed: 0']
del edgelist_snowball_2_4_2['Unnamed: 0']
del edgelist_snowball_2_4_3['Unnamed: 0']
del edgelist_snowball_2_4_4['Unnamed: 0']
del edgelist_snowball_2_4_5['Unnamed: 0']

edgelist_snowball_2 = pd.concat([edgelist_snowball_2_1, edgelist_snowball_2_2, edgelist_snowball_2_3, edgelist_snowball_2_4_1, edgelist_snowball_2_4_2, edgelist_snowball_2_4_3, edgelist_snowball_2_4_4, edgelist_snowball_2_4_5], ignore_index=True,axis=0)
edgelist_snowball_2.to_csv('edgelist_snowball_2.csv') # Save it to file

In [ ]:
len(edgelist_snowball_2)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


2059761

In [ ]:
# START FINAL ROUND FROM HERE
edgelist_all_FPE_papers = pd.read_csv('edgelist_all_FPE_papers.csv')
edgelist_snowball_1 = pd.read_csv('edgelist_snowball_1.csv')
edgelist_snowball_2 = pd.read_csv('edgelist_snowball_2.csv')
del edgelist_all_FPE_papers['Unnamed: 0']
del edgelist_snowball_1['Unnamed: 0']
del edgelist_snowball_2['Unnamed: 0']
FPE_DOI = edgelist_all_FPE_papers['source'].unique()
snowball_1_DOI = edgelist_snowball_1['source'].unique()
snowball_2_DOI = edgelist_snowball_2['source'].unique()
snowball_2_DOI_target = edgelist_snowball_2['target'].unique()
DOI_peniultimate = np.concatenate([FPE_DOI, snowball_1_DOI, snowball_2_DOI])
snowball_2_DOI_target = list(snowball_2_DOI_target)
DOI_peniultimate = list(DOI_peniultimate)
DOI_to_prepare = list(set(list(snowball_2_DOI_target)).difference(DOI_peniultimate)) # List the ones that were not repeated before

In [ ]:
df_1, df_2, df_3, df_4, df_5, df_6, df_7, df_8, df_9, df_10, df_11, df_12 = np.array_split(DOI_to_prepare, 12)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
df_1_1, df_1_2, df_1_3, df_1_4, df_1_5 = np.array_split(df_12, 5)

In [ ]:
# FINAL ROUND OF FINDING REFERENCES
total_start = time.time()

edgelist_all_FPE_papers = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame

for i in range(0,len(df_1_1)):
    try:
        result = Crossref().works(query = f'{df_1_1[i]}') ######### MODIFY THIS THIS
        time.sleep(0.025)
        #Check if the result contains the 'message' and 'items' keys before accessing them
        if 'message' in result and 'items' in result['message'] and len(result['message']['items']) > 0:
            result = result['message']['items'][0]
            edgelist_single_paper = pd.DataFrame(columns=['source','target', 'target_title']) # Your existing DataFrame
            references_doi = []
            references_title = []
            # Check if the result contains the 'reference' key before accessing it
            if 'reference' in result:
                for j in range(0,len(result['reference'])):
                    try:
                        # Ensure both DOI and title are extracted before appending
                        doi = result['reference'][j].get('DOI')
                        title = result['reference'][j].get('article-title')
                        if doi and title: # Append only if both values are present
                            references_doi.append(doi)
                            references_title.append(title)
                    except:
                        continue
            references_df = pd.DataFrame({'source': [result['DOI']] * len(references_doi), 'target': references_doi, 'target_title': references_title})
            edgelist_single_paper = pd.concat([edgelist_single_paper, references_df], ignore_index=True)
            edgelist_all_FPE_papers = pd.concat([edgelist_all_FPE_papers, edgelist_single_paper], ignore_index=True)
        else:
            print(f"Skipping record {i} due to missing data in Crossref response.")
    except Exception:
        pass  # This was not indented properly

edgelist_all_FPE_papers.to_csv('edgelist_snowball_3_12_1.csv')

total_end = time.time()
total_elapsed_seconds = total_end - total_start
total_elapsed_time = format_timespan(total_elapsed_seconds)
send_push_final(total_elapsed_time)

In [ ]:
# By here you should have all of the data downloaded.
# Now re-load it, compile it to a single edgelist and then another file linking title and doi
FPE_core = pd.read_csv('FPE_core.csv')
edgelist_all_FPE_papers = pd.read_csv('edgelist_all_FPE_papers.csv')
edgelist_snowball_1 = pd.read_csv('edgelist_snowball_1.csv')
edgelist_snowball_2 = pd.read_csv('edgelist_snowball_2.csv')
del edgelist_all_FPE_papers['Unnamed: 0']
del edgelist_snowball_1['Unnamed: 0']
del edgelist_snowball_2['Unnamed: 0']

In [ ]:
# Final step of the snowball was large, was scrapped in chunks.
# Now put all the chunks into a single folder and compile them to a single file
# df = pd.DataFrame()
path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/sum"
files = Path(path).glob('*.csv')
dfs = list()
for f in files:
    data = pd.read_csv(f)
    # .stem is method for pathlib objects to get the filename w/o the extension
    data['file'] = f.stem
    dfs.append(data)
edgelist_snowball_3 = pd.concat(dfs, ignore_index=True)
del edgelist_snowball_3['file']

In [ ]:
len(FPE_core_list_plus_4)

685733

In [ ]:
# Now compile list of all papers and then also an edgelist
FPE_core_list = FPE_core[['DOI', 'Title']]
FPE_core_list_plus_1 = edgelist_all_FPE_papers[['target', 'target_title']]
FPE_core_list_plus_1 = FPE_core_list_plus_1.rename(columns={'target': 'DOI', 'target_title': 'Title'})
FPE_core_list_plus_2 = edgelist_snowball_1[['target', 'target_title']]
FPE_core_list_plus_2 = FPE_core_list_plus_2.rename(columns={'target': 'DOI', 'target_title': 'Title'})
FPE_core_list_plus_3 = edgelist_snowball_2[['target', 'target_title']]
FPE_core_list_plus_3 = FPE_core_list_plus_3.rename(columns={'target': 'DOI', 'target_title': 'Title'})
FPE_core_list_plus_4 = edgelist_snowball_3[['target', 'target_title']]
FPE_core_list_plus_4 = FPE_core_list_plus_4.rename(columns={'target': 'DOI', 'target_title': 'Title'})

# This is a list of all unique papers. It's uncut, i.e. it contains many papers that are not in the scope of forest policy and economics
all_papers_uncut = pd.concat([FPE_core_list, FPE_core_list_plus_1, FPE_core_list_plus_2, FPE_core_list_plus_3, FPE_core_list_plus_4])
all_papers_uncut = all_papers_uncut.drop_duplicates(subset=['DOI', 'Title'], keep='first') # Remove duplicates
edgelist_all = pd.concat([edgelist_all_FPE_papers, edgelist_snowball_1, edgelist_snowball_2, edgelist_snowball_3])
del edgelist_all['target_title']
del edgelist_all['Unnamed: 0']
edgelist_all = edgelist_all.drop_duplicates(subset=['source', 'target'], keep='first') # Remove duplicates


In [ ]:
edgelist_all.to_csv('edgelist_all.csv')
all_papers_uncut.to_csv('all_papers_uncut.csv')

In [ ]:
# START FROM HERE

edgelist_all = pd.read_csv('edgelist_all.csv')
all_papers_uncut = pd.read_csv('all_papers_uncut.csv')
del edgelist_all['Unnamed: 0']

In [ ]:
# Saving and loading list https://stackoverflow.com/questions/27745500/how-to-save-a-list-to-a-file-and-read-it-as-a-list-type
community = [list(sub_community) for sub_community in community]
with open("community.json", 'w') as f:
    json.dump(community, f, indent=2)

In [ ]:
# https://networkx.org/documentation/stable/reference/algorithms/community.html
# https://github.com/esclear/louvain-leiden?tab=readme-ov-file#sample-notebooks
# https://www.nature.com/articles/s41598-019-41695-z

In [ ]:
# https://medium.com/@adiyaochir/getting-started-with-python-igraph-a-comprehensive-guide-with-examples-3386820990c1
# https://igraph.org/python/tutorial/0.9.6/visualisation.html

# CHUNK WITH DEBUGGING / DATA QUALITY CONTROL

In [ ]:
# 28.3.2026.

In [ ]:
# This chunk is to make an index table / data-frame from all the papers in the pkl / list objects
# The table is first formed from the 'references' pkl files
# All papers should exists both in 'references' and 'cited by' files
# in 'cited by' there's somewhat smaller number of publications, as not all of them have eid / SCOPUS's ID

In [ ]:
# FUNCTIONS

def make_df_info(file_name_string, references_file_name):
    table_data = []

    for i, entry in enumerate(references_file_name):
        if entry is None: # Skip if the entry is None
            print(f"Warning: Skipping None entry at index {i} in {file_name_string}.")
            continue
        row = {}
        abs_resp = entry.get('abstracts-retrieval-response', {})
        item = abs_resp.get('item', {})
        coredata = abs_resp.get('coredata', {})
        bibrecord = item.get('bibrecord', {})
        head = bibrecord.get('head', {})
        meta = item.get('xocs:meta', {})

        # 1. funding-agency-matched-string
        funding_list = meta.get('xocs:funding-list', {}).get('xocs:funding', [])
        if not isinstance(funding_list, list): funding_list = [funding_list] # Handle single item not in list
        row['funding-agency-matched-string'] = "; ".join(filter(None, [
            f.get('xocs:funding-agency-matched-string', '') for f in funding_list
        ]))

        # 2. xocs:funding-id
        funding_ids = []
        for f in funding_list:
            f_id = f.get('xocs:funding-id', '')
            if isinstance(f_id, list):
                funding_ids.append("; ".join(filter(None, [str(x) for x in f_id])))
            else:
                funding_ids.append(str(f_id))
        row['xocs:funding-id'] = "; ".join(filter(None, funding_ids))

        # Author Group related fields (country, @country, organization_$, affiliation-id_@afid, given_name, @author-instance-id, @auid)
        author_group_list = head.get('author-group', [])
        if not isinstance(author_group_list, list): author_group_list = [author_group_list]

        all_countries = []
        all_at_countries = []
        all_organizations = []
        all_affiliation_ids = []
        all_given_names = []
        all_author_instance_ids = []
        all_auids = []

        for ag in author_group_list:
            affiliation_list = ag.get('affiliation', [])
            if not isinstance(affiliation_list, list): affiliation_list = [affiliation_list]
            for aff in affiliation_list:
                all_countries.append(aff.get('country', ''))
                all_at_countries.append(aff.get('@country', ''))
                org = aff.get('organization', {})
                if isinstance(org, dict):
                    all_organizations.append(org.get('$', ''))
                elif isinstance(org, list):
                    # Filter out None values and ensure they are dictionaries before calling .get('$')
                    all_organizations.extend([o.get('$', '') for o in org if isinstance(o, dict)])

                # FIX START: Handle affiliation-id as a list of dictionaries
                aff_id_entries = aff.get('affiliation-id', [])
                if not isinstance(aff_id_entries, list):
                    aff_id_entries = [aff_id_entries]
                for aff_id_item in aff_id_entries:
                    if isinstance(aff_id_item, dict):
                        all_affiliation_ids.append(aff_id_item.get('@afid', ''))
                # FIX END

            author_list = ag.get('author', [])
            if not isinstance(author_list, list): author_list = [author_list]
            for auth in author_list:
                all_given_names.append(auth.get('ce:indexed-name', ''))
                all_author_instance_ids.append(auth.get('@author-instance-id', ''))
                all_auids.append(auth.get('@auid', ''))

        row['country'] = "; ".join(filter(None, all_countries))
        row['@country'] = "; ".join(filter(None, all_at_countries))
        row['organization_$'] = "; ".join(filter(None, all_organizations))
        row['affiliation-id_@afid'] = "; ".join(filter(None, all_affiliation_ids))
        row['given_name'] = "; ".join(filter(None, all_given_names))
        row['@author-instance-id'] = "; ".join(filter(None, all_author_instance_ids))
        row['@auid'] = "; ".join(filter(None, all_auids))

        # 10. citation-title
        row['citation-title'] = head.get('citation-title', '')

        # 11. abstract
        row['abstract'] = coredata.get('dc:description', '')

        # 12. author_keywords
        author_keywords_container = head.get('citation-info', {}).get('author-keywords', {})
        author_keyword_list = author_keywords_container.get('author-keyword', [])
        if not isinstance(author_keyword_list, list): author_keyword_list = [author_keyword_list]
        row['author_keywords'] = "; ".join(filter(None, [
            k.get('$', '') for k in author_keyword_list
        ]))

        # 13. journal
        source = head.get('source', {})
        row['journal'] = str(source.get('sourcetitle', '')) # Cast to string to avoid ArrowTypeError

        # 14. year
        publication_date = source.get('publicationdate', {})
        row['year'] = publication_date.get('year', '')

        # 15. issn
        issn_data = source.get('issn', []) # Ensure it's treated as a list, default to empty list
        if not isinstance(issn_data, list): issn_data = [issn_data] # Handle single item not in list
        row['issn'] = "; ".join(filter(None, [
            issn.get('$', '') for issn in issn_data if isinstance(issn, dict)
        ]))

        # Grant related fields (grant-acronym, grant-agency_@iso-code, grant-agency-id)
        grant_list_container = head.get('grantlist', {})
        grant_list = grant_list_container.get('grant', [])
        if not isinstance(grant_list, list): grant_list = [grant_list]

        all_grant_acronyms = []
        all_grant_agency_iso_codes = []
        all_grant_agency_ids = []

        for grant_item in grant_list:
            all_grant_acronyms.append(grant_item.get('grant-acronym', ''))
            grant_agency_data = grant_item.get('grant-agency', {})

            if isinstance(grant_agency_data, dict):
                all_grant_agency_iso_codes.append(grant_agency_data.get('@iso-code', ''))
            else:
                # If it's a string or other non-dict type, append an empty string as it doesn't have '@iso-code'
                all_grant_agency_iso_codes.append('')

            all_grant_agency_ids.append(grant_item.get('grant-agency-id', ''))

        row['grant-acronym'] = "; ".join(filter(None, all_grant_acronyms))
        row['grant-agency_@iso-code'] = "; ".join(filter(None, all_grant_agency_iso_codes))
        row['grant-agency-id'] = "; ".join(filter(None, all_grant_agency_ids))

        # 19. eid
        row['eid'] = coredata.get('eid', '')

        # 20. doi
        row['doi'] = coredata.get('prism:doi', '')

        # 21. file_name
        row['references_file_name'] = file_name_string # Constant - manually change to the loaded file!

        # 22. file_index
        row['references_file_index'] = i

        # 24.for EID placeholder columns
        row['cited_by_file_name'] = ''
        row['cited_by_file_index'] = ''

        table_data.append(row)


    df_references_info = pd.DataFrame(table_data)
    return df_references_info


# This function corrects the index designation to a file with JSON list of references
def check_index_designation_references(filtered_df, references_json_1):
    # Initialize a flag column to track updates
    filtered_df['references_index_updated'] = False

    # 1. Create a dictionary for fast DOI lookup from references_json_X... provisionally labeled 1
    doi_to_index_map = {}
    for i, entry in enumerate(references_json_1):
        if entry is None:
            continue
        try:
            abstracts_retrieval_response = entry.get('abstracts-retrieval-response')
            if abstracts_retrieval_response is None:
                continue
            coredata = abstracts_retrieval_response.get('coredata')
            if coredata is None:
                continue
            doi_in_list = coredata.get('prism:doi')
            if doi_in_list:
                doi_to_index_map[doi_in_list] = i
        except KeyError:
            continue

    # 2. Iterate through the filtered DataFrame and update the 'references_file_index' using the map
    for idx, row in filtered_df.iterrows():
        target_doi = row['doi']
        found_index = doi_to_index_map.get(target_doi, -1)

        # If a new index is found, update the main DataFrame and set the flag
        if found_index != -1:
            filtered_df.loc[idx, 'references_file_index'] = found_index
            filtered_df.loc[idx, 'references_index_updated'] = True
        else:
            # Handle cases where the DOI from the DataFrame is not found in the list
            print(f"Warning: DOI {target_doi} from df_references_info_loaded not found in references_json.")
    print("References file indices in df_references_info_loaded updated based on references_json.")
    filtered_df = filtered_df.drop(columns=['references_index_updated'])
    return filtered_df


# Make edgelist. DOIs of target papers are from DOI identifier of references. If it does not exist, then it's parsed out from full reference
def make_edgelist_from_references(df_info_partial, references_json):
    edgelist_rows = []
    for i in range(len(df_info_partial)):
        sender_index = df_info_partial.index[i]
        sender_title = df_info_partial.loc[sender_index, 'citation-title']
        sender_doi = df_info_partial.loc[sender_index, 'doi']
        sender_eid = df_info_partial.loc[sender_index, 'eid']
        index_of_reference = (df_info_partial['references_file_index'].iloc[i])

        # Safely get the list of references for the current focal paper
        current_references_entry = references_json[index_of_reference]
        abstracts_resp = current_references_entry.get('abstracts-retrieval-response', {})
        item = abstracts_resp.get('item', {})
        bibrecord = item.get('bibrecord', {})
        # Ensure tail is a dictionary. If bibrecord.get('tail') is None, use an empty dict.
        tail = bibrecord.get('tail') or {}
        bibliography = tail.get('bibliography', {})
        references_list = bibliography.get('reference', [])

        if not isinstance(references_list, list):
            references_list = [references_list] # Handle case where it might be a single dict

        for ref in references_list: # Iterate through each reference dictionary
            ref_fulltext_string = ref.get('ref-fulltext', '') # Use .get() to avoid KeyError

            receiver_doi = None # Initialize
            receiver_title = None # Initialize

            # --- New logic: Try to get DOI from ref-info['refd-itemidlist']['itemid'] first ---
            ref_info = ref.get('ref-info', {})
            refd_itemidlist = ref_info.get('refd-itemidlist', {})
            itemid_list = refd_itemidlist.get('itemid', [])

            if not isinstance(itemid_list, list):
                itemid_list = [itemid_list]

            for itemid in itemid_list:
                if isinstance(itemid, dict) and itemid.get('@idtype') == 'DOI':
                    receiver_doi = itemid.get('$', '')
                    break # Found a DOI, no need to check other itemids

            # If DOI was not found in refd-itemidlist, fall back to extracting from ref_fulltext_string
            if not receiver_doi and ref_fulltext_string:
                last_comma_index = ref_fulltext_string.rfind(',')
                if last_comma_index != -1:
                    extracted_string = ref_fulltext_string[last_comma_index + 1:].strip()
                    extracted_string = extracted_string.replace(' ', '')
                    if extracted_string.endswith('.'):
                        extracted_string = extracted_string[:-1]
                    receiver_doi = extracted_string

            # Get receiver title
            ref_title_data = ref_info.get('ref-title', {})
            receiver_title = ref_title_data.get('ref-titletext', ref_fulltext_string if not receiver_doi else '')

            # Append to edgelist_rows. Use None for receiver_index as it's not determined here.
            edgelist_rows.append([sender_index, sender_title, sender_doi, sender_eid, None, receiver_title, receiver_doi, ref_fulltext_string])

    edgelist_df = pd.DataFrame(edgelist_rows, columns=['source_index', 'source_title', 'source_doi', 'source_eid', 'target_index', 'target_title', 'target_doi', 'target_fulltext_string'])
    return edgelist_df


In [ ]:
# Some functions to clean full reference to get the DOI - for ABSTRACT searches that don't have DOI of references by key

def contains_numbers(s):
    if pd.isna(s): # Handle NaN values
        return False
    return any(char.isdigit() for char in str(s))

def clean_doi(doi):
    if pd.isna(doi):
        return None
    s_doi = str(doi)
    # Condition 1: shorter than 8 characters
    if len(s_doi) < 6:
        return None
    # Condition 2: more than 4 letters in a row
    if re.search(r'[a-zA-Z]{4,}', s_doi):
        return None
    return doi


def further_clean_doi(doi):
    if pd.isna(doi):
        return None
    s_doi = str(doi)
    # Condition 1: Starts with '%'
    if s_doi.startswith('%'):
        return None
    # Condition 2: Ends with 'xls'
    if s_doi.endswith('.xls'):
        return None
    # Condition 3: Entire expression is within brackets '()'
    if s_doi.startswith('(') and s_doi.endswith(')'):
        return None
    return doi


def remove_page_range_patterns(doi):
    if pd.isna(doi):
        return None
    s_doi = str(doi)
    # Pattern: one to three-4 numbers, followed by an en dash or hyphen, followed by one to three-4 numbers
    # Examples: 257–272, 123-456
    if re.fullmatch(r'\d{1,3}[–-]\d{1,3}', s_doi):   ## !!!!! Expand this for 4-digits!
        return None
    if re.fullmatch(r'\d{1,4}[–-]\d{1,4}', s_doi):   ## !!!!! Expand this for 4-digits!
        return None
    return doi

# Apply the new cleaning function to the 'target_doi' column


def clean_title(title):
    if pd.isna(title):
        return ''
    # Convert to string, lowercase, remove special characters, and strip whitespace
    cleaned_title = str(title).lower()
    cleaned_title = re.sub(r'[^a-z0-9\s]', '', cleaned_title) # Keep only alphanumeric and spaces
    cleaned_title = ' '.join(cleaned_title.split()) # Replace multiple spaces with single space and strip
    return cleaned_title

In [ ]:
with open('references_json_8.pkl', 'rb') as f: # CHANGE THE FILE NAME DESIGNATION HERE!!!!
    references_json_8 = pickle.load(f)

In [ ]:
#assign metadata to the newly downloaded files and then concat them to the main metadata file
df_info = make_df_info('references_json_8', references_json_8)
df_info = check_index_designation_references(df_info, references_json_8)
# Join the files
df_info_full = pd.concat([df_info_full, df_info])
df_info_full = df_info_full.reset_index(drop=True)
# Save it
df_info_full.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')

References file indices in df_references_info_loaded updated based on references_json.


In [ ]:
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')


In [ ]:
# Assign which file and endex is for 'cited by'

# 1. Create a mapping from EID to a list of indices from cited_by_json_1
eid_to_indices = defaultdict(list)
for i, item in enumerate(cited_by_json_1):
    try:
        # Extract the 'eid' from the current item
        raw_search_term = item['search-results']['opensearch:Query']['@searchTerms']
        eid = raw_search_term.replace('REFEID(', '').replace(')', '')
        eid_to_indices[eid].append(str(i)) # Store index as string for later joining
    except (KeyError, TypeError):
        # Handle cases where the structure might be missing, or 'eid' cannot be extracted
        print(f"Warning: Could not extract EID for item at index {i} in cited_by_json_1. Skipping.")
        continue

# Convert lists of indices to semicolon-separated strings
eid_to_indexed_strings = {eid: ";".join(indices) for eid, indices in eid_to_indices.items()}

# 2. Update df_references_info_loaded based on this mapping
# Identify rows where 'cited_by_file_name' is empty
empty_cited_by_mask = df_references_info_loaded['cited_by_file_name'] == ''

# Create temporary Series for easier mapping, applying only to relevant EIDs
temp_cited_by_file_name = df_references_info_loaded.loc[empty_cited_by_mask, 'eid'].map(
    lambda eid: 'cited_by_json_3_3' if eid in eid_to_indexed_strings else '' # Change the file name here!
)
temp_cited_by_file_index = df_references_info_loaded.loc[empty_cited_by_mask, 'eid'].map(
    lambda eid: eid_to_indexed_strings.get(eid, '')
)

# Assign these temporary Series to the DataFrame columns, but only for the empty rows
df_references_info_loaded.loc[empty_cited_by_mask, 'cited_by_file_name'] = temp_cited_by_file_name
df_references_info_loaded.loc[empty_cited_by_mask, 'cited_by_file_index'] = temp_cited_by_file_index

print("Updated 'cited_by_file_name' and 'cited_by_file_index' columns in df_references_info_loaded.")

Updated 'cited_by_file_name' and 'cited_by_file_index' columns in df_references_info_loaded.


In [ ]:
# Check if all the files are correctly assigned
# One of the tests is this - to see if a single paper exists across multiple list items, i.e. those papers which have been cited a lot / over 200 times
df_references_info_loaded[df_references_info_loaded['cited_by_file_index'].str.contains(';', na=False)]

In [ ]:
# Define the path to your Google Drive (assuming it's already mounted)
google_drive_path = '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/'

# Ensure the directory exists
os.makedirs(google_drive_path, exist_ok=True)

# Save as Parquet (for large datasets)
output_parquet_path = os.path.join(google_drive_path, 'df_info_full.parquet')
df_references_info_loaded.to_parquet(output_parquet_path, index=False)

In [ ]:
# 31.3.2026

In [ ]:
# The last columns in the table say which file and which index the data of the paper is in
# The designation to files is correct, but for some of the files the index designation is not - it's 2 indexes higer
# Now I check this and rectify this

In [ ]:
# Assign which file and endex is for 'cited by'

# 1. Create a mapping from EID to a list of indices from cited_by_json_1
eid_to_indices = defaultdict(list)
for i, item in enumerate(cited_by_json_1):
    try:
        # Extract the 'eid' from the current item
        raw_search_term = item['search-results']['opensearch:Query']['@searchTerms']
        eid = raw_search_term.replace('REFEID(', '').replace(')', '')
        eid_to_indices[eid].append(str(i)) # Store index as string for later joining
    except (KeyError, TypeError):
        # Handle cases where the structure might be missing, or 'eid' cannot be extracted
        print(f"Warning: Could not extract EID for item at index {i} in cited_by_json_1. Skipping.")
        continue

# Convert lists of indices to semicolon-separated strings
eid_to_indexed_strings = {eid: ";".join(indices) for eid, indices in eid_to_indices.items()}

# 2. Update df_references_info_loaded based on this mapping
# Identify rows where 'cited_by_file_name' is empty
empty_cited_by_mask = df_references_info_loaded['cited_by_file_name'] == ''

# Create temporary Series for easier mapping, applying only to relevant EIDs
temp_cited_by_file_name = df_references_info_loaded.loc[empty_cited_by_mask, 'eid'].map(
    lambda eid: 'cited_by_json_3_3' if eid in eid_to_indexed_strings else '' # Change the file name here!!!!
)
temp_cited_by_file_index = df_references_info_loaded.loc[empty_cited_by_mask, 'eid'].map(
    lambda eid: eid_to_indexed_strings.get(eid, '')
)

# Assign these temporary Series to the DataFrame columns, but only for the empty rows
df_references_info_loaded.loc[empty_cited_by_mask, 'cited_by_file_name'] = temp_cited_by_file_name
df_references_info_loaded.loc[empty_cited_by_mask, 'cited_by_file_index'] = temp_cited_by_file_index

print("Updated 'cited_by_file_name' and 'cited_by_file_index' columns in df_references_info_loaded.")

Updated 'cited_by_file_name' and 'cited_by_file_index' columns in df_references_info_loaded.


In [ ]:
# Check if all the files are correctly assigned
# One of the tests is this - to see if a single paper exists across multiple list items, i.e. those papers which have been cited a lot / over 200 times
df_references_info_loaded[df_references_info_loaded['cited_by_file_index'].str.contains(';', na=False)]

In [ ]:
### END OF CHUNK WITH DEBUGGING / DATA QUALITY CONTROL ####

In [ ]:
# 3.4.2026

# making of edgelist from references

In [ ]:
# making of edgelist - first on references
# Many references are not recognized. These will be added later on

In [ ]:
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')

In [ ]:
with open('references_json_8.pkl', 'rb') as f: # CHANGE THE FILE NAME DESIGNATION HERE!!!!
    references_json = pickle.load(f)

In [ ]:
# Split the df_info_full based on pkl lists with references
df_info_partial = df_info_full[df_info_full['references_file_name'] =='references_json_8'] # CHANGE THE FILE NAME DESIGNATION HERE!!!!

In [ ]:
edgelist_df = make_edgelist_from_references(df_info_partial, references_json)

In [ ]:
references_json[2002]

In [ ]:
edgelist_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_7.parquet')

In [ ]:
# Put all them togheter
edgelist_df_1 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_1.parquet')
edgelist_df_2 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_2.parquet')
edgelist_df_3 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_3.parquet')
edgelist_df_4 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_4.parquet')
edgelist_df_5 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_5.parquet')
edgelist_df_6 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_6.parquet')
edgelist_df_7 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_7.parquet')

In [ ]:
edgelist_df = pd.concat([edgelist_df_1, edgelist_df_2, edgelist_df_3, edgelist_df_4, edgelist_df_5, edgelist_df_6, edgelist_df_7])

In [ ]:
edgelist_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references.parquet')

In [ ]:
# Now some cleaning of the edgelist

In [ ]:
# remove duplicate rows
edgelist_df.drop_duplicates(inplace=True)
# apply DOI cleaning
# Not all references have DOI
# With this you remove the DOIs which are not real DOI
edgelist_df['target_doi'] = edgelist_df['target_doi'].apply(lambda x: x if contains_numbers(x) else None)
edgelist_df['target_doi'] = edgelist_df['target_doi'].apply(clean_doi)
edgelist_df['target_doi'] = edgelist_df['target_doi'].apply(further_clean_doi)
edgelist_df['target_doi'] = edgelist_df['target_doi'].apply(remove_page_range_patterns)

In [ ]:
edgelist_df.columns

Index(['source_index', 'source_title', 'source_doi', 'source_eid',
       'target_index', 'target_title', 'target_doi', 'target_fulltext_string'],
      dtype='object')

In [ ]:
# Manually inspect the strange DOIs and remove them
sorted_target_doi_df = edgelist_df['target_doi'].sort_values().reset_index(drop=True)
sorted_target_doi_df = pd.DataFrame(sorted_target_doi_df)
print(sorted_target_doi_df.head())

      target_doi
0    345:101-118
1  36(9):1340-61
2  43(3):377-403
3   46(5):715-31
4     5(1):11-26


In [ ]:
sorted_target_doi_df.iloc[30000:42000,:]

,target_doi


In [ ]:
# Get the DOIs from the first XYZ rows of sorted_target_doi_df
dois_to_replace = sorted_target_doi_df.head(41997)['target_doi'].dropna().unique() # Change here manually in each file!
# Convert to a set for efficient lookup
dois_to_replace_set = set(dois_to_replace)
# Assign None to matching values in edgelist_df['target_doi']
edgelist_df.loc[edgelist_df['target_doi'].isin(dois_to_replace_set), 'target_doi'] = None


In [ ]:
edgelist_df['target_eid'] = ''

In [ ]:
# Match now by title
# Apply cleaning to titles in both DataFrames
edgelist_df['cleaned_target_title'] = edgelist_df['target_title'].apply(clean_title)
df_info_full['cleaned_citation_title'] = df_info_full['citation-title'].apply(clean_title)

# Create a temporary DataFrame from df_info_full for title-based merge
df_info_for_merge_by_title = df_info_full[['doi', 'eid', 'citation-title', 'cleaned_citation_title']].copy()
df_info_for_merge_by_title = df_info_for_merge_by_title.rename(columns={
    'doi': 'matched_target_doi_by_title',
    'eid': 'matched_target_eid_by_title',
    'citation-title': 'matched_target_raw_title_by_title',
    'cleaned_citation_title': 'cleaned_target_title' # This is the key for merging
})
df_info_for_merge_by_title['target_index_from_info_by_title'] = df_info_full.index

# Perform a left merge to bring in potential matches by title for remaining empty entries
edgelist_df_temp_title = edgelist_df[edgelist_df['target_eid'].eq('') | edgelist_df['target_doi'].eq('')].merge(
    df_info_for_merge_by_title, on='cleaned_target_title', how='left', suffixes=('', '_by_title'))

# Update original edgelist_df using values from edgelist_df_temp_title
# only for rows where original target_eid or target_doi was empty
mask_to_update = edgelist_df['target_eid'].eq('') | edgelist_df['target_doi'].eq('')

edgelist_df.loc[mask_to_update, 'target_eid'] = edgelist_df.loc[mask_to_update, 'target_eid'].fillna(edgelist_df_temp_title['matched_target_eid_by_title'])
edgelist_df.loc[mask_to_update, 'target_doi'] = edgelist_df.loc[mask_to_update, 'target_doi'].fillna(edgelist_df_temp_title['matched_target_doi_by_title'])
edgelist_df.loc[mask_to_update, 'target_title'] = edgelist_df.loc[mask_to_update, 'target_title'].fillna(edgelist_df_temp_title['matched_target_raw_title_by_title'])
edgelist_df.loc[mask_to_update, 'target_index'] = edgelist_df.loc[mask_to_update, 'target_index'].fillna(edgelist_df_temp_title['target_index_from_info_by_title']).astype('Int64')

# Convert remaining NaN/empty strings back to empty strings for consistency if desired
edgelist_df['target_eid'] = edgelist_df['target_eid'].fillna('')
edgelist_df['target_doi'] = edgelist_df['target_doi'].fillna('')
edgelist_df['target_title'] = edgelist_df['target_title'].fillna('')

# Clean up temporary columns
edgelist_df = edgelist_df.drop(columns=['cleaned_target_title'])
del df_info_full['cleaned_citation_title'] # Clean up the column from df_info_full if not needed later

/tmp/ipykernel_6171/1603148421.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  edgelist_df.loc[mask_to_update, 'target_index'] = edgelist_df.loc[mask_to_update, 'target_index'].fillna(edgelist_df_temp_title['target_index_from_info_by_title']).astype('Int64')


In [ ]:
# This section is to lookup DOI at Crossref
### Here add the DOI's of target_doi for papers that were not recognized in SCOPUS, as based on full reference search in Crossref

In [ ]:
missing_nan_count = edgelist_df['target_doi'].isna().sum()
missing_empty_string_count = (edgelist_df['target_doi'] == '').sum()

total_missing_count = missing_nan_count + missing_empty_string_count
share_missing_doi = total_missing_count / len(edgelist_df)

print(f"Share of missing values in 'target_doi' (including empty strings): {share_missing_doi:.2%}")

Share of missing values in 'target_doi' (including empty strings): 38.40%


In [ ]:
# Separate the papers that don't have a DOI
missing_DOI_df = edgelist_df[edgelist_df['target_doi'].isna() | (edgelist_df['target_doi'] == '')]
missing_DOI_df = missing_DOI_df.drop(columns=['source_index', 'source_title', 'source_doi', 'source_eid'])
missing_DOI_df = missing_DOI_df.drop_duplicates(subset=['target_title'])

In [ ]:
cr = Crossref(timeout=60)  # Increase the timeout to 60 seconds

chunk_size = 1000
for start_index in range(0, len(missing_DOI_df), chunk_size):
    end_index = min(start_index + chunk_size, len(missing_DOI_df))
    for i in range(start_index, end_index):
            try:
                # Add a small delay to avoid overwhelming the API
                time.sleep(0.5)  # Wait for 0.5 seconds between requests
                result = cr.works(query = missing_DOI_df.iloc[i,3])
                if 'message' in result and 'items' in result['message'] and result['message']['items']:
                    DOI_TO_SEARCH = result['message']['items'][0]['DOI']
                    missing_DOI_df.iloc[i,2] = DOI_TO_SEARCH
                else:
                    # Handle cases where the query doesn't return expected structure or items
                    print(f"No results found for query: {missing_DOI_df.iloc[i,3]}")
                    continue
            except requests.exceptions.Timeout:
                print(f"Request timed out for query: {missing_DOI_df.iloc[i,3]}")
                # You might want to implement a retry mechanism here
                continue
            except Exception as e:
                print(f"An error occurred at index {i}: {e}")
                continue
    # Save the DataFrame periodically
    missing_DOI_df.to_csv('missing_DOI_df_references_1.csv')

In [ ]:
missing_DOI_df_references_1.to_csv('missing_DOI_df_references_1.csv')

In [ ]:
# End of Crossref lookup section

In [ ]:
# 8.4.2026

# Load the files
missing_DOI_df_references_1 = pd.read_csv('missing_DOI_Crossref_df_references_1.csv')
missing_DOI_df_references_1 = missing_DOI_df_references_1.drop(columns=['Unnamed: 0'])

In [ ]:
# Create a mapping from 'target_fulltext_string' to 'target_doi' from missing_DOI_df_references_1
# Assuming missing_DOI_df_references_1 has 'target_fulltext_string' and 'target_doi' columns
doi_mapping = missing_DOI_df_references_1.set_index('target_fulltext_string')['target_doi'].to_dict()
# Replace empty strings with actual NaN for consistent handling across the entire DataFrame
edgelist_df['target_doi'] = edgelist_df['target_doi'].replace('', np.nan)
# Create a new column with potential DOIs from the mapping for all rows
edgelist_df['mapped_doi'] = edgelist_df['target_fulltext_string'].map(doi_mapping)
# Update 'target_doi': prefer the mapped_doi if available, otherwise keep the original target_doi
edgelist_df['target_doi'] = edgelist_df['mapped_doi'].fillna(edgelist_df['target_doi'])
# Drop the temporary mapped_doi column
edgelist_df = edgelist_df.drop(columns=['mapped_doi'])
print("Updated 'target_doi' in edgelist_df based on matching 'target_fulltext_string' for all rows.")

Updated 'target_doi' in edgelist_df based on matching 'target_fulltext_string' for all rows.


In [ ]:
edgelist_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references.parquet')

In [ ]:
###################
################################################

In [ ]:
# See the share of missing data
missing_nan_count = edgelist_df_references_1['target_doi'].isna().sum()
missing_empty_string_count = (edgelist_df_references_1['target_doi'] == '').sum()

total_missing_count = missing_nan_count + missing_empty_string_count
share_missing_doi = total_missing_count / len(edgelist_df_references_1)

print(f"Share of missing values in 'target_doi' (including empty strings): {share_missing_doi:.2%}")

Share of missing values in 'target_doi' (including empty strings): 10.19%


In [ ]:
# Map indexes of existing targets, i.e. check for which paper you already have data on

# Create a mapping from DOI to index in df_info_full
doi_to_index_map = pd.Series(df_info_full.index.values, index=df_info_full['doi']).to_dict()

# Identify rows in edgelist_df_references_1 where target_index is NaN
mask_nan_target_index = edgelist_df_references_1['target_index'].isna()

# Map the target_doi to the corresponding index from df_info_full
edgelist_df_references_1.loc[mask_nan_target_index, 'target_index'] = \
    edgelist_df_references_1.loc[mask_nan_target_index, 'target_doi'].map(doi_to_index_map)

# Convert target_index to integer type, allowing for NaN (Int64)
edgelist_df_references_1['target_index'] = edgelist_df_references_1['target_index'].astype('Int64')

print("Updated 'target_index' based on DOI matching with df_info_full.")
# Display the first few rows to show the updated target_index
display(edgelist_df_references_1[mask_nan_target_index].head())

Updated 'target_index' based on DOI matching with df_info_full.


,source_index,source_title,source_doi,source_eid,target_index,target_title,target_doi,target_fulltext_string,target_eid
1,0,Envisioning future forested landscapes in Swed...,10.1016/j.forpol.2016.07.010,2-s2.0-84984677390,<NA>,Methodologies in Action Research,NaN,"Aagaard Nielsen, K., Nielsen, B.S., Methodolog...",NaN
2,0,Envisioning future forested landscapes in Swed...,10.1016/j.forpol.2016.07.010,2-s2.0-84984677390,<NA>,Citizens' Initiatives for Democratic Nature Ma...,NaN,"Aagaard Nielsen, K., Nielsen, B.S., Citizens' ...",NaN
3,0,Envisioning future forested landscapes in Swed...,10.1016/j.forpol.2016.07.010,2-s2.0-84984677390,<NA>,NaN,NaN,"Aagaard Nielsen, K., Svensson, L., (eds.) Acti...",NaN
10,0,Envisioning future forested landscapes in Swed...,10.1016/j.forpol.2016.07.010,2-s2.0-84984677390,<NA>,"Frame Analysis, Place Perceptions and the Poli...",NaN,"Beland Lindahl, K., Frame Analysis, Place Perc...",NaN
11,0,Envisioning future forested landscapes in Swed...,10.1016/j.forpol.2016.07.010,2-s2.0-84984677390,<NA>,Place perceptions and controversies over fores...,10.1080/1523908x.2012.753316,"Beland Lindahl, K., Baker, S., Waldenström, C....",NaN


In [ ]:
edgelist_df_references_1.to_csv('edgelist_df_references_1.csv')

In [ ]:
# Now you see that there are referenced papers on which you don't have info on.
# Here you list these DOI's - > Then get it via SCOPUS!
missing_references_1_SCOPUS = edgelist_df_references_1[['target_index', 'target_title', 'target_doi', 'target_fulltext_string', 'target_eid']]
missing_references_1_SCOPUS = missing_references_1_SCOPUS[missing_references_1_SCOPUS['target_index'].isna()]
missing_references_1_SCOPUS = missing_references_1_SCOPUS.drop(columns=['target_index'])
missing_references_1_SCOPUS = missing_references_1_SCOPUS[missing_references_1_SCOPUS['target_doi'].notna() & (missing_references_1_SCOPUS['target_doi'] != '')]
missing_references_1_SCOPUS = missing_references_1_SCOPUS.drop_duplicates()
scopus_urls = get_scopus_urls_ABSTRACT(missing_references_1_SCOPUS['target_doi'], api_key)
scopus_urls_df = pd.DataFrame({'scopus_urls': scopus_urls})
scopus_urls_df.to_csv('missing_references_1_SCOPUS.txt', sep='\t', index=False, header=False)

In [ ]:
# 10.4.2026
edgelist_df_references_1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_1.csv')
edgelist_df_references_1 = edgelist_df_references_1.drop(columns=['Unnamed: 0'])

In [ ]:
# Load all the files from SCOPUS REFERENCES (that produced hits) to a single list

file_path = "/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/missing_references_1_SCOPUS"
files = list_files(file_path)
# Initialize json_all_data as a list to store the combined data
json_all_data = []
if files:
  for file_name in files:
    data_dir_path = os.path.join(file_path, file_name)
    json_data = open_and_read_json(data_dir_path)
    json_all_data.append(json_data)
else:
  print(f"Directory '{file_path}' not found or empty.")

# Save the list
with open('references_json_7.pkl', 'wb') as f:
  pickle.dump(json_all_data, f)

UnicodeDecodeError: Retrying with 'latin-1' for '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/missing_references_1_SCOPUS/.DS_Store'.
Error: Could not read '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/missing_references_1_SCOPUS/.DS_Store' with 'latin-1' either: Expecting value: line 1 column 1 (char 0)


In [ ]:
# Load the files
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')
with open('references_json_7.pkl', 'rb') as f:
    references_json_7 = pickle.load(f)

In [ ]:
#assign metadata to the newly downloaded files and then concat them to the main metadata file
df_info = make_df_info('references_json_7', references_json_7)
df_info = check_index_designation_references(df_info, references_json_7)
# Join the files
df_info_full = pd.concat([df_info_full, df_info])
df_info_full = df_info_full.reset_index(drop=True)
# Save it
df_info_full.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')

In [ ]:
df_info_full.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')

In [ ]:
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')


In [ ]:
# Map index values from df_info_full to edgelist_df_references_1
index_map = pd.Series(df_info_full.index, index=df_info_full['doi']).to_dict()
edgelist_df_references_1['source_index'] = edgelist_df_references_1['source_doi'].map(index_map).fillna(edgelist_df_references_1['source_index'])
edgelist_df_references_1['target_index'] = edgelist_df_references_1['target_doi'].map(index_map).fillna(edgelist_df_references_1['target_index'])
edgelist_df_references_1['source_index'] = edgelist_df_references_1['source_index'].astype('Int64')
edgelist_df_references_1['target_index'] = edgelist_df_references_1['target_index'].astype('Int64')

In [ ]:
# Save edgelist_df_references_1
edgelist_df_references_1.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_1.parquet')

In [ ]:
# get all unique target_fulltext_string from this filtered edgelist

In [ ]:
missing_target_index_df = edgelist_df_references_1[edgelist_df_references_1['target_index'].isna()]
unique_target_fulltext_strings = missing_target_index_df['target_fulltext_string'].unique()

print(f"Number of unique target fulltext strings with missing target_index: {len(unique_target_fulltext_strings)}")
print("First 5 unique target fulltext strings:")
print(unique_target_fulltext_strings[:5])

Number of unique target fulltext strings with missing target_index: 41593
First 5 unique target fulltext strings:
['Aagaard Nielsen, K., Nielsen, B.S., Methodologies in Action Research. Aagaard Nielsen, K., Svensson, L., (eds.) Action and Interactive Research : Beyond Practice and Theory, 2006, Shaker Publishing, Maastricht.'
 "Aagaard Nielsen, K., Nielsen, B.S., Citizens' Initiatives for Democratic Nature Management and Community Development - Reflecting on Danish Experiences. Hansen, H.P., Nielsen, B.S., Sriskandarajah, N., Gunnarsson, E., (eds.) Commons, Sustainability, Democratization - Action Research and the Basic Renewal of Society, 2016, Routlegde, New York."
 'Aagaard Nielsen, K., Svensson, L., (eds.) Action and Interactive Research : Beyond Practice and Theory, 2006, Shaker Publishing, Maastricht.'
 'Beland Lindahl, K., Frame Analysis, Place Perceptions and the Politics of Natural Resource Management - Exploring a Forest Policy Controversy in Sweden. (Dr. Thesis), 2008, Swedi

In [ ]:
missing_DOI_Crossref_df_references_1_2 = pd.DataFrame({'target_fulltext_string': unique_target_fulltext_strings})
missing_DOI_Crossref_df_references_1_2['target_doi'] = '' # Initialize a new column for the DOIs to be found

41593

In [ ]:
combined_missing_DOI_Crossref_df.to_csv('missing_DOI_Crossref_df_references_1.csv')

In [ ]:
missing_DOI_Crossref_df_references_1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/missing_DOI_Crossref_df_references_1.csv')
missing_DOI_Crossref_df_references_1 = missing_DOI_Crossref_df_references_1.drop(columns=['Unnamed: 0'])

In [ ]:
edgelist_df_references_1 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_1.parquet')

In [ ]:
# Add missing DOIs to targeted / referenced papers
# I already have these in
# Create a mapping from 'target_fulltext_string' to 'target_doi' from missing_DOI_Crossref_df_references_1
doi_mapping = missing_DOI_Crossref_df_references_1.set_index('target_fulltext_string')['target_doi'].to_dict()

# Identify rows in edgelist_df_references_1 where 'target_doi' is NaN
nan_doi_mask = edgelist_df_references_1['target_doi'].isna()

# Apply the mapping to fill NaN values in 'target_doi'
edgelist_df_references_1.loc[nan_doi_mask, 'target_doi'] = \
    edgelist_df_references_1.loc[nan_doi_mask, 'target_fulltext_string'].map(doi_mapping)

print("Updated 'target_doi' in edgelist_df_references_1 based on matching 'target_fulltext_string'.")

Updated 'target_doi' in edgelist_df_references_1 based on matching 'target_fulltext_string'.


In [ ]:
# And again go to SCOPUS for the final round
missing_references_1_SCOPUS = edgelist_df_references_1[edgelist_df_references_1['target_index'].isna()]
missing_references_1_SCOPUS = missing_references_1_SCOPUS[['target_fulltext_string', 'target_doi']]
missing_references_1_SCOPUS = missing_references_1_SCOPUS.drop_duplicates(subset=['target_doi'])
scopus_urls = get_scopus_urls_ABSTRACT(missing_references_1_SCOPUS['target_doi'], api_key)
scopus_urls_df = pd.DataFrame({'scopus_urls': scopus_urls})
scopus_urls_df.to_csv('missing_references_1_2_SCOPUS.txt', sep='\t', index=False, header=False)

In [ ]:
cr = Crossref(timeout=60)  # Increase the timeout to 60 seconds

chunk_size = 1000
for start_index in range(0, len(missing_DOI_Crossref_df_references_1_2), chunk_size):
    end_index = min(start_index + chunk_size, len(missing_DOI_Crossref_df_references_1_2))
    for i in range(start_index, end_index):
            try:
                # Add a small delay to avoid overwhelming the API
                time.sleep(0.5)  # Wait for 0.5 seconds between requests
                result = cr.works(query = missing_DOI_Crossref_df_references_1_2.iloc[i,0])
                if 'message' in result and 'items' in result['message'] and result['message']['items']:
                    DOI_TO_SEARCH = result['message']['items'][0]['DOI']
                    missing_DOI_Crossref_df_references_1_2.iloc[i,1] = DOI_TO_SEARCH
                else:
                    # Handle cases where the query doesn't return expected structure or items
                    print(f"No results found for query: {missing_DOI_Crossref_df_references_1_2.iloc[i,0]}")
                    continue
            except requests.exceptions.Timeout:
                print(f"Request timed out for query: {missing_DOI_Crossref_df_references_1_2.iloc[i,0]}")
                # You might want to implement a retry mechanism here
                continue
            except Exception as e:
                print(f"An error occurred at index {i}: {e}")
                continue
    # Save the DataFrame periodically
    missing_DOI_df.to_csv('missing_DOI_Crossref_df_references_1_2_X.csv') # CHANGE HERE IS YOU HAVE SPLIT THE FILE / DF

In [ ]:
# Now cycle through them all, i.e. all the files with references and make a single edgelist
# Then get find DOI for all references that do not have it
# Then get JSON SCOPUS references files for all these papers as based on DOI
# then lookup EID and target_index as based on existing entries in df_info_full

In [ ]:
# Then continue with cited_by_json_1

In [ ]:
google_drive_path = '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/'

# Ensure the directory exists
os.makedirs(google_drive_path, exist_ok=True)

# Save as Parquet (for large datasets)
output_parquet_path = os.path.join(google_drive_path, 'edgelist_df_references_2.parquet') # Changed filename to 1 as it's the first file processed from references
edgelist_df.to_parquet(output_parquet_path, index=False)
print(f"DataFrame saved as Parquet to: {output_parquet_path}")

DataFrame saved as Parquet to: /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_2.parquet


# Edgelist from citations

In [ ]:
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')

In [ ]:
# Repeat this and the next cell for 'cited by' snowball 1 and 2 and 3
with open('cited_by_json_3_4.pkl', 'rb') as f: # CHANGE THE FILE NAME DESIGNATION HERE!!!!
    cited_by_json_1 = pickle.load(f)

In [ ]:
edgelist_data = []

for item_index, item in enumerate(cited_by_json_1):
    if item is None:
        print(f"Warning: Skipping None item at index {item_index} in cited_by_json_1.")
        continue
    try:
        # Extract target_eid (focal publication EID)
        search_results = item.get('search-results', {})
        opensearch_query = search_results.get('opensearch:Query', {})
        search_terms_string = opensearch_query.get('@searchTerms')

        if search_terms_string is None:
            print(f"Skipping item at index {item_index}: '@searchTerms' key is missing or None.")
            continue

        target_eid = search_terms_string.replace('REFEID(', '').replace(')', '')

        # Iterate through papers that cite the focal publication
        citing_papers_entries = search_results.get('entry', [])
        if not citing_papers_entries:
            # print(f"Skipping item at index {item_index}: 'entry' key is missing or empty.")
            continue

        for entry in citing_papers_entries:
            source_title = entry.get('dc:title', '')
            source_doi = entry.get('prism:doi', '')
            source_eid = entry.get('eid', '')

            edgelist_data.append({
                'target_eid': target_eid,
                'source_title': source_title,
                'source_doi': source_doi,
                'source_eid': source_eid
            })
    except Exception as e: # Catch a broader exception for unexpected issues
        print(f"An unexpected error occurred for item at index {item_index}: {e}")
        continue

edgelist_cited_by = pd.DataFrame(edgelist_data)
print(edgelist_cited_by.head())

           target_eid                                       source_title  \
0  2-s2.0-33646201969  The impact of converting rice cultivation to g...   
1  2-s2.0-33646201969  Divergent responses of cropland soil available...   
2  2-s2.0-33646201969  Basin characteristics determine longitudinal d...   
3  2-s2.0-33646201969  Integrated innovation and application of green...   
4  2-s2.0-33646201969  Changes in soil organic carbon and phosphorus ...   

                      source_doi           source_eid  
0    10.1016/j.still.2025.106658  2-s2.0-105005167301  
1    10.1016/j.still.2025.106654  2-s2.0-105004656359  
2  10.1016/j.ecolind.2025.114309  2-s2.0-105022158154  
3        10.15302/J-FASE-2025636  2-s2.0-105020446315  
4   10.1016/j.geodrs.2025.e00950  2-s2.0-105000267858  


In [ ]:
edgelist_cited_by

,target_eid,source_title,source_doi,source_eid
0,2-s2.0-33646201969,The impact of converting rice cultivation to g...,10.1016/j.still.2025.106658,2-s2.0-105005167301
1,2-s2.0-33646201969,Divergent responses of cropland soil available...,10.1016/j.still.2025.106654,2-s2.0-105004656359
2,2-s2.0-33646201969,Basin characteristics determine longitudinal d...,10.1016/j.ecolind.2025.114309,2-s2.0-105022158154
3,2-s2.0-33646201969,Integrated innovation and application of green...,10.15302/J-FASE-2025636,2-s2.0-105020446315
4,2-s2.0-33646201969,Changes in soil organic carbon and phosphorus ...,10.1016/j.geodrs.2025.e00950,2-s2.0-105000267858
...,...,...,...,...
3252914,2-s2.0-0027668609,Equationless and equation-based trend models o...,10.1016/j.techfore.2016.07.031,2-s2.0-84999816135
3252915,2-s2.0-0027668609,Fuzzy model of relationship among economic per...,10.11118/actaun201260040071,2-s2.0-84864254319
3252916,2-s2.0-0027668609,Empirical specification of cost reductions ass...,10.1016/j.forpol.2008.02.004,2-s2.0-50549096302
3252917,2-s2.0-0027668609,A fuzzy pooling of investment cost knowledge,10.1016/0925-5273(95)00155-7,2-s2.0-0030172593


In [ ]:
# With this you can check is the edgelist correct
df_info_full[df_info_full['eid'] == edgelist_cited_by['target_eid'][0]].iloc[0,:]

In [ ]:
google_drive_path = '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/'
# Ensure the directory exists
os.makedirs(google_drive_path, exist_ok=True)
# Save as Parquet (for large datasets)
output_parquet_path = os.path.join(google_drive_path, 'edgelist_cited_by_3_3.parquet') # Changed filename!!
edgelist_cited_by.to_parquet(output_parquet_path, index=False)
print(f"DataFrame saved as Parquet to: {output_parquet_path}")

DataFrame saved as Parquet to: /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_3_3.parquet


In [ ]:
edgelist_cited_by_1_and_2 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_1_and_2.parquet')
edgelist_cited_by_3_1 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_3_1.parquet')
edgelist_cited_by_3_2 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_3_2.parquet')
edgelist_cited_by_3_3 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_3_3.parquet')

In [ ]:
edgelist_cited_by_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet')

In [ ]:
len(edgelist_cited_by_uncut)

5628195

In [ ]:
edgelist_cited_by_uncut = pd.concat([edgelist_cited_by_1_and_2, edgelist_cited_by_3_1, edgelist_cited_by_3_2, edgelist_cited_by_3_3])
edgelist_cited_by_uncut = edgelist_cited_by_uncut.drop_duplicates()

In [ ]:
edgelist_cited_by_uncut = pd.concat([edgelist_cited_by_uncut, edgelist_cited_by])
edgelist_cited_by_uncut = edgelist_cited_by_uncut.drop_duplicates()

In [ ]:
len(edgelist_cited_by_uncut)

8805160

In [ ]:
google_drive_path = '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/'
# Ensure the directory exists
os.makedirs(google_drive_path, exist_ok=True)
# Save as Parquet (for large datasets)
output_parquet_path = os.path.join(google_drive_path, 'edgelist_cited_by_uncut.parquet') # Changed filename t
edgelist_cited_by_uncut.to_parquet(output_parquet_path, index=False)
print(f"DataFrame saved as Parquet to: {output_parquet_path}")

DataFrame saved as Parquet to: /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet


In [ ]:
edgelist_cited_by_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet')

In [ ]:
edgelist_df_references_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet')

In [ ]:
# Prepare df_info_full for merging, selecting only relevant columns
df_info_for_merge = df_info_full[['eid', 'citation-title', 'doi']].copy()
df_info_for_merge = df_info_for_merge.rename(columns={'citation-title': 'target_title', 'doi': 'target_doi'})

# Perform a left merge to add 'target_title' and 'target_doi' to edgelist_cited_by_uncut
edgelist_cited_by_uncut = pd.merge(
    edgelist_cited_by_uncut,
    df_info_for_merge,
    left_on='target_eid',
    right_on='eid',
    how='left'
)

# Drop the redundant 'eid' column from the merge if it's not needed (it's duplicated by target_eid)
edgelist_cited_by_uncut = edgelist_cited_by_uncut.drop(columns=['eid'])

print(edgelist_cited_by_uncut.head())

In [ ]:
edgelist_cited_by_uncut

,source_title,source_doi,source_eid,target_title_x,target_doi_x,target_eid,target_title_y,target_doi_y
0,The Role of Stakeholders in Managing Social Fo...,10.29244/jpsl.15.1.77,2-s2.0-85214667779,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007
1,The Role of Stakeholders in Managing Social Fo...,10.29244/jpsl.15.1.77,2-s2.0-85214667779,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007
2,Community engagement in the governance of Camb...,10.1016/j.forpol.2024.103386,2-s2.0-85212311678,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007
3,Community engagement in the governance of Camb...,10.1016/j.forpol.2024.103386,2-s2.0-85212311678,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007
4,A different dimension in deforestation and for...,10.1016/j.landusepol.2024.107086,2-s2.0-85184747956,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007
...,...,...,...,...,...,...,...,...
9184376,Equationless and equation-based trend models o...,10.1016/j.techfore.2016.07.031,2-s2.0-84999816135,None,None,2-s2.0-0027668609,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W
9184377,Fuzzy model of relationship among economic per...,10.11118/actaun201260040071,2-s2.0-84864254319,None,None,2-s2.0-0027668609,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W
9184378,Empirical specification of cost reductions ass...,10.1016/j.forpol.2008.02.004,2-s2.0-50549096302,None,None,2-s2.0-0027668609,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W
9184379,A fuzzy pooling of investment cost knowledge,10.1016/0925-5273(95)00155-7,2-s2.0-0030172593,None,None,2-s2.0-0027668609,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W


In [ ]:
edgelist_cited_by_uncut = edgelist_cited_by_uncut[[
    'source_title', 'source_doi', 'source_eid',
    'target_title_y', 'target_doi_y', 'target_eid'
]]
edgelist_cited_by_uncut

,source_title,source_doi,source_eid,target_title_y,target_doi_y,target_eid
0,The Role of Stakeholders in Managing Social Fo...,10.29244/jpsl.15.1.77,2-s2.0-85214667779,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373
1,The Role of Stakeholders in Managing Social Fo...,10.29244/jpsl.15.1.77,2-s2.0-85214667779,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373
2,Community engagement in the governance of Camb...,10.1016/j.forpol.2024.103386,2-s2.0-85212311678,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373
3,Community engagement in the governance of Camb...,10.1016/j.forpol.2024.103386,2-s2.0-85212311678,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373
4,A different dimension in deforestation and for...,10.1016/j.landusepol.2024.107086,2-s2.0-85184747956,The participation of stakeholders in the polic...,10.1016/j.forpol.2017.05.007,2-s2.0-85019681373
...,...,...,...,...,...,...
9184376,Equationless and equation-based trend models o...,10.1016/j.techfore.2016.07.031,2-s2.0-84999816135,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W,2-s2.0-0027668609
9184377,Fuzzy model of relationship among economic per...,10.11118/actaun201260040071,2-s2.0-84864254319,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W,2-s2.0-0027668609
9184378,Empirical specification of cost reductions ass...,10.1016/j.forpol.2008.02.004,2-s2.0-50549096302,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W,2-s2.0-0027668609
9184379,A fuzzy pooling of investment cost knowledge,10.1016/0925-5273(95)00155-7,2-s2.0-0030172593,A fuzzy upgrading of integrated vague manageri...,10.1016/0925-5273(93)90069-W,2-s2.0-0027668609


In [ ]:
edgelist_cited_by_uncut = edgelist_cited_by_uncut.rename(columns={'target_title_y': 'target_title', 'target_doi_y': 'target_doi'})

In [ ]:
edgelist_cited_by_uncut.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet', index=False)

In [ ]:
# Cut to sampling frame, remove duplicates
edgelist_cited_by_cut = edgelist_cited_by_uncut[edgelist_cited_by_uncut['source_doi'].isin(df_info_full['doi'])]
edgelist_cited_by_cut = edgelist_cited_by_cut.drop_duplicates()
google_drive_path = '/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/'
os.makedirs(google_drive_path, exist_ok=True)
output_parquet_path = os.path.join(google_drive_path, 'edgelist_cited_by_cut.parquet') # Changed filename t
edgelist_cited_by_cut.to_parquet(output_parquet_path, index=False)
print(f"DataFrame saved as Parquet to: {output_parquet_path}")

DataFrame saved as Parquet to: /content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_cut.parquet


In [ ]:
# Assign EID to targeted paper in edgelist_df_references_uncut
df_info_for_merge_eid = df_info_full[['eid', 'doi']].copy()
df_info_for_merge_eid = df_info_for_merge_eid.rename(columns={'doi': 'target_doi'})

# Perform a left merge to add 'eid' to edgelist_df_references_uncut
edgelist_df_references_uncut = pd.merge(
    edgelist_df_references_uncut,
    df_info_for_merge_eid,
    on='target_doi',
    how='left',
    suffixes=('', '_from_info')
)

# Rename the merged 'eid' column to 'target_eid'
edgelist_df_references_uncut = edgelist_df_references_uncut.rename(columns={'eid_from_info': 'target_eid'})

print(edgelist_df_references_uncut.head())

# Drop duplicates
edgelist_df_references_uncut = edgelist_df_references_uncut.drop_duplicates()

# Save it
edgelist_df_references_uncut.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet', index=False)


In [ ]:
# Remove extra columns
edgelist_df_references_uncut = edgelist_df_references_uncut.drop(columns=['source_index', 'target_index', 'target_fulltext_string'])
# rename edgelist_df_references_uncut column eid to target_eid
edgelist_df_references_uncut = edgelist_df_references_uncut.rename(columns={'eid': 'target_eid'})
# re-order columns to match the edgelist based on citations
edgelist_df_references_uncut = edgelist_df_references_uncut[[
    'source_title', 'source_doi', 'source_eid',
    'target_title', 'target_doi', 'target_eid'
]]
# Save it (again)
edgelist_df_references_uncut.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet', index=False)

In [ ]:
# Cut to sampling frame
edgelist_df_references_cut = edgelist_df_references_uncut[
    edgelist_df_references_uncut['target_doi'].isin(df_info_full['doi'])
]

In [ ]:
edgelist_df_references_cut.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_cut.parquet', index=False)

In [ ]:
sampling_frame_320k = pd.concat([edgelist_cited_by_cut, edgelist_df_references_cut])
sampling_frame_320k = sampling_frame_320k.drop_duplicates()
sampling_frame_320k = sampling_frame_320k.reset_index(drop=True)

In [ ]:
sampling_frame_320k.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_sampling_frame_320k.parquet', index=False)

# TO improve the edgelist_df_references_uncut

In [ ]:
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')
df_info_full = df_info_full.reset_index(drop=True)

In [ ]:
df_info_full.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet', index=False)

In [ ]:
edgelist_df_references_1 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_1.parquet')
edgelist_df_references_2 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_2.parquet')
edgelist_df_references_3 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_3.parquet')
edgelist_df_references_4 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_4.parquet')
edgelist_df_references_5 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_5.parquet')
edgelist_df_references_6 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_6.parquet')
edgelist_df_references_7 = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_7.parquet')

In [ ]:
edgelist_df_references_uncut = pd.concat([edgelist_df_references_1, edgelist_df_references_2, edgelist_df_references_3, edgelist_df_references_4, edgelist_df_references_5, edgelist_df_references_6, edgelist_df_references_7])
edgelist_df_references_uncut = edgelist_df_references_uncut.reset_index(drop=True)


In [ ]:
edgelist_df_references_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet')

In [ ]:
# Check if there is a match between the targeted papers in edgelist_df_references_uncut and df_info_full and assign these DOIs
# Then assign these to target_doi


def clean_title(title):
    if pd.isna(title):
        return ''
    cleaned_title = str(title).lower()
    cleaned_title = re.sub(r'[^a-z0-9\s]', '', cleaned_title) # Keep only alphanumeric and spaces
    cleaned_title = ' '.join(cleaned_title.split()) # Replace multiple spaces with single space and strip
    return cleaned_title

# 1. Create a cleaned title column in edgelist_df_references_uncut for matching
edgelist_df_references_uncut['cleaned_target_title'] = edgelist_df_references_uncut['target_title'].apply(clean_title)

# 2. Prepare df_info_full for matching by cleaning citation titles
df_info_full_for_matching = df_info_full[['doi', 'citation-title']].copy()
df_info_full_for_matching['cleaned_citation_title'] = df_info_full_for_matching['citation-title'].apply(clean_title)

# Drop rows where cleaned_citation_title is empty or duplicates to ensure a clean merge
df_info_full_for_matching = df_info_full_for_matching[df_info_full_for_matching['cleaned_citation_title'] != '']
df_info_full_for_matching = df_info_full_for_matching.drop_duplicates(subset=['cleaned_citation_title'], keep='first')

# 3. Perform a left merge to bring in DOIs from df_info_full based on exact title match
# This creates a temporary DataFrame with matched DOIs
merged_temp_df = pd.merge(
    edgelist_df_references_uncut,
    df_info_full_for_matching[['doi', 'cleaned_citation_title']],
    left_on='cleaned_target_title',
    right_on='cleaned_citation_title',
    how='left',
    suffixes=('', '_from_info')
)

# 4. Assign the matched DOIs to 'target_doi_from_df_full'
edgelist_df_references_uncut['target_doi_from_df_full'] = merged_temp_df['doi']

# Display the first few rows with the new column
print(edgelist_df_references_uncut[['target_fulltext_string', 'target_doi_from_df_full']].head())

# Also print the count of non-null values in the new column
print(f"Number of DOIs assigned to 'target_doi_from_df_full': {edgelist_df_references_uncut['target_doi_from_df_full'].count()}")

# Clean up the temporary 'cleaned_target_title' column from edgelist_df_references_uncut
edgelist_df_references_uncut = edgelist_df_references_uncut.drop(columns=['cleaned_target_title'])

                              target_fulltext_string  \
0  Aagaard Nielsen, K., Sustainability and democr...   
1  Aagaard Nielsen, K., Nielsen, B.S., Methodolog...   
2  Aagaard Nielsen, K., Nielsen, B.S., Citizens' ...   
3  Aagaard Nielsen, K., Svensson, L., (eds.) Acti...   
4  Andersson, K., Angelstam, P., Elbakidze, M., A...   

        target_doi_from_df_full  
0      10.1177/1476750305052139  
1                           NaN  
2                           NaN  
3                           NaN  
4  10.1080/02827581.2012.723740  
Number of DOIs assigned to 'target_doi_from_df_full': 3272355


In [ ]:
edgelist_df_references_uncut.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet', index=False)

In [ ]:
edgelist_df_references_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet')

In [ ]:
# Clean DOIs
edgelist_df_references_uncut['target_doi'] = edgelist_df_references_uncut['target_doi'].apply(lambda x: x if contains_numbers(x) else None)
edgelist_df_references_uncut['target_doi'] = edgelist_df_references_uncut['target_doi'].apply(clean_doi)
edgelist_df_references_uncut['target_doi'] = edgelist_df_references_uncut['target_doi'].apply(further_clean_doi)
edgelist_df_references_uncut['target_doi'] = edgelist_df_references_uncut['target_doi'].apply(remove_page_range_patterns)

In [ ]:
# Add doi from missing references
missing_DOI_Crossref_df_references_1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/missing_DOI_Crossref_df_references_1.csv')
missing_DOI_Crossref_df_references_1 = missing_DOI_Crossref_df_references_1.drop(columns=['Unnamed: 0'])
# Create a mapping from 'target_fulltext_string' to 'target_doi' from missing_DOI_Crossref_df_references_1
doi_mapping = missing_DOI_Crossref_df_references_1.set_index('target_fulltext_string')['target_doi'].to_dict()

# Identify rows in edgelist_df_references_uncut where 'target_doi' is NaN or empty string
nan_or_empty_doi_mask = edgelist_df_references_uncut['target_doi'].isna() | (edgelist_df_references_uncut['target_doi'] == '')

# Apply the mapping to fill NaN/empty values in 'target_doi'
edgelist_df_references_uncut.loc[nan_or_empty_doi_mask, 'target_doi'] = \
    edgelist_df_references_uncut.loc[nan_or_empty_doi_mask, 'target_fulltext_string'].map(doi_mapping)

print("Updated 'target_doi' in edgelist_df_references_uncut based on matching 'target_fulltext_string'.")

In [ ]:
# Copy the matching DOIs from df_info_full to target_doi

# First, ensure that empty strings in 'target_doi' are treated as NaN for consistent filling
edgelist_df_references_uncut['target_doi'] = edgelist_df_references_uncut['target_doi'].replace('', np.nan)
# Prioritize non-NaN values from 'target_doi_from_df_full' over existing 'target_doi'
edgelist_df_references_uncut['target_doi'] = edgelist_df_references_uncut['target_doi_from_df_full'].combine_first(edgelist_df_references_uncut['target_doi'])

In [ ]:
edgelist_df_references_uncut = edgelist_df_references_uncut.drop(columns=['target_doi_from_df_full'])

In [ ]:
# Save
edgelist_df_references_uncut.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet', index=False)

In [ ]:
edgelist_df_references_uncut

In [ ]:
df_info_full_dois = set(df_info_full['doi'].dropna().unique())
edgelist_source_dois = set(edgelist_df_references_uncut['source_doi'].dropna().unique())

dois_not_in_edgelist = list(df_info_full_dois - edgelist_source_dois)

print(f"Number of DOIs in df_info_full not in edgelist_df_references_uncut['source_doi']: {len(dois_not_in_edgelist)}")
print(dois_not_in_edgelist[:10]) # Print first 10 for brevity

Number of DOIs in df_info_full not in edgelist_df_references_uncut['source_doi']: 2287
['10.1111/j.1467-9299.1995.tb00850.x', '10.1201/b17154', '10.1139/x88-199', '10.1016/j.wdp.2021.100322', '10.14214/aff.7674', '10.1068/d090451', '10.1108/meq.2012.08323daa.014', '10.4271/520107', '10.2307/1879675', '10.1016/j.meatsci.2015.05.001']


In [ ]:
scopus_urls = get_scopus_urls_ABSTRACT(dois_not_in_edgelist, api_key)
scopus_urls_df = pd.DataFrame({'scopus_urls': scopus_urls})
scopus_urls_df.to_csv('scopus_urls_missing_abstract_references.txt', sep='\t', index=False, header=False)

In [ ]:
##### scopus_urls_missing_abstract_references are not downloaded!!!!!!!!!!!!!!!!!!

In [ ]:
# Same for 'cited-by' now

####### DO NOT FORGET TO MAKE FULL QUERY FOR PAPERS CITED OVER 200 TIMES!!!!

In [ ]:
edgelist_cited_by_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet')

In [ ]:
df_info_full_dois = set(df_info_full['doi'].dropna().unique())
edgelist_target_cited_by_dois = set(edgelist_cited_by_uncut['target_doi'].dropna().unique())

dois_not_in_df_info_full = list(df_info_full_dois - edgelist_target_cited_by_dois)
len(dois_not_in_df_info_full)

40379

In [ ]:
eids_from_dois_not_in_cited_by = df_info_full[df_info_full['doi'].isin(dois_not_in_df_info_full)]['eid'].tolist()
print(f"Number of EIDs found: {len(eids_from_dois_not_in_cited_by)}")
print(eids_from_dois_not_in_cited_by[:10]) # Displaying the first 10 EIDs

Number of EIDs found: 42365
['2-s2.0-85033804402', '2-s2.0-84863774613', '2-s2.0-84925739514', '2-s2.0-85091594292', '2-s2.0-0035733237', '2-s2.0-35348879715', '2-s2.0-34948845082', '2-s2.0-86000296325', '2-s2.0-85049263514', '2-s2.0-48249126474']


In [ ]:
unique_eids_from_dois_not_in_cited_by = set(eids_from_dois_not_in_cited_by)

In [ ]:
len(unique_eids_from_dois_not_in_cited_by)

40380

In [ ]:
api_key = 'XYZ' # Replace with your actual API key
scopus_links = get_scopus_urls_STANDARD(unique_eids_from_dois_not_in_cited_by, api_key)
scopus_links_df = pd.DataFrame({'scopus_urls': scopus_links})

print(scopus_links[:5]) # Print the first 5 links as a sample

In [ ]:
scopus_links_df.to_csv('scopus_urls_missing_EID_cited_by.txt', sep='\t', index=False, header=False)

In [ ]:
edgelist_cited_by_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet')

In [ ]:
#################
############ Now to cut the data to the sample
#####################

In [ ]:
df_info_full = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/df_info_full.parquet')

In [ ]:
edgelist_df_references_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_df_references_uncut.parquet')

In [ ]:
edgelist_cited_by_uncut = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cited_by_uncut.parquet')

In [ ]:
edgelist_df_references_cut = edgelist_df_references_uncut[
    edgelist_df_references_uncut['target_doi'].isin(df_info_full['doi'])]

In [ ]:
len(edgelist_df_references_cut) / len(edgelist_df_references_uncut)

0.1372456101742792

In [ ]:
edgelist_cited_by_cut = edgelist_cited_by_uncut[
    edgelist_cited_by_uncut['source_doi'].isin(df_info_full['doi'])]

In [ ]:
len(edgelist_cited_by_cut) / len(edgelist_cited_by_uncut)

0.36328534280100094

In [ ]:
# First, ensure edgelist_df_references_cut has a 'target_eid' column.
# We can get this by merging with df_info_full based on target_doi.

# Create a temporary DataFrame from df_info_full containing 'doi' and 'eid'
df_info_for_eid_merge = df_info_full[['doi', 'eid']].rename(columns={'doi': 'target_doi', 'eid': 'target_eid'})

# Perform a left merge to add 'target_eid' to edgelist_df_references_cut
edgelist_df_references_cut = pd.merge(
    edgelist_df_references_cut,
    df_info_for_eid_merge,
    on='target_doi',
    how='left',
    suffixes=('', '_from_info_full')
)

# If there's already a target_eid column (e.g., from previous merges), ensure we keep the correct one.
# The merge will add `target_eid` (or `target_eid_from_info_full` if there's a conflict).
# We want the newly merged one, or if it already existed and was correct, that one.
# For simplicity, if a target_eid_from_info_full exists, we'll use it.
# If `target_eid` was initially empty or NaN, fill it with the merged `target_eid_from_info_full`.

# Drop target_index and target_fulltext_string if they are no longer needed
edgelist_df_references_cut = edgelist_df_references_cut.drop(columns=['target_index', 'target_fulltext_string'])

# Now select and reorder the columns as requested
edgelist_df_references_cut = edgelist_df_references_cut[['source_title', 'source_doi', 'source_eid', 'target_title', 'target_doi', 'target_eid']]

print(edgelist_df_references_cut.head())

In [ ]:
edgelist_sampling_frame_draft = pd.concat([edgelist_cited_by_cut, edgelist_df_references_cut])

In [ ]:
edgelist_sampling_frame_draft.to_parquet('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/edgelist_cut_draft.parquet', index=False)

In [ ]:
# 24.4.2026

In [ ]:
file_path = 'scopus_urls_missing_EID_cited_by.txt'

urls_list = []
with open(file_path, 'r') as f:
    for line in f:
        urls_list.append(line.strip()) # .strip() removes leading/trailing whitespace, including newline characters

print(f"Loaded {len(urls_list)} URLs from {file_path}")
print(f"First 5 URLs: {urls_list[:5]}")

In [ ]:
import pickle

file_path_pkl = 'scopus_urls_missing_EID_cited_by.pkl'

try:
    with open(file_path_pkl, 'rb') as f:
        scopus_urls_missing_EID_cited_by = pickle.load(f)
    print(f"Successfully loaded '{file_path_pkl}' as a list.")
except FileNotFoundError:
    print(f"Error: The file '{file_path_pkl}' was not found. Please ensure it exists in the current working directory.")
except Exception as e:
    print(f"An error occurred while loading the file '{file_path_pkl}': {e}")

Successfully loaded 'scopus_urls_missing_EID_cited_by.pkl' as a list.


In [ ]:
file_path_pkl = 'urls_not_in_scopus_list.pkl'

try:
    with open(file_path_pkl, 'rb') as f:
        urls_not_in_scopus_list = pickle.load(f)
    print(f"Successfully loaded '{file_path_pkl}' as a list.")
except FileNotFoundError:
    print(f"Error: The file '{file_path_pkl}' was not found. Please ensure it exists in the current working directory.")
except Exception as e:
    print(f"An error occurred while loading the file '{file_path_pkl}': {e}")

Successfully loaded 'urls_not_in_scopus_list.pkl' as a list.


In [ ]:
len(scopus_urls_missing_EID_cited_by)

40390

In [ ]:
cleaned_list = []
for item in scopus_urls_missing_EID_cited_by:
    if item is not None:
        # Attempt to access a key to ensure it's a dictionary (JSON object)
        try:
            # Accessing 'search-results' as seen in the trace for valid items
            _ = item['search-results']
            cleaned_list.append(item)
        except (TypeError, KeyError):
            # This item is not a dictionary or lacks expected structure, so skip it
            continue

scopus_urls_missing_EID_cited_by = cleaned_list
print(f"Cleaned list now contains {len(scopus_urls_missing_EID_cited_by)} valid items.")

Cleaned list now contains 40378 valid items.


In [ ]:
import re

# 1. Extract REFEIDs from urls_list
refeids_from_urls_list = set()
for url in urls_list:
    match = re.search(r'REFEID\((.*?)\)', url)
    if match:
        refeids_from_urls_list.add(match.group(1))

# 2. Extract REFEIDs from scopus_urls_missing_EID_cited_by
refeids_from_scopus_list = set()
for item in scopus_urls_missing_EID_cited_by:
    try:
        search_terms = item['search-results']['opensearch:Query']['@searchTerms']
        match = re.search(r'REFEID\((.*?)\)', search_terms)
        if match:
            refeids_from_scopus_list.add(match.group(1))
    except (KeyError, TypeError):
        # Handle cases where the structure might be missing or corrupted
        continue

# 3. Find REFEIDs unique to urls_list
unique_refeids = refeids_from_urls_list - refeids_from_scopus_list

# 4. Filter the original urls_list to get URLs corresponding to unique_refeids
urls_not_in_scopus_list = []
for url in urls_list:
    match = re.search(r'REFEID\((.*?)\)', url)
    if match and match.group(1) in unique_refeids:
        urls_not_in_scopus_list.append(url)

print(f"Number of unique REFEIDs in urls_list: {len(refeids_from_urls_list)}")
print(f"Number of unique REFEIDs in scopus_urls_missing_EID_cited_by: {len(refeids_from_scopus_list)}")
print(f"Number of REFEIDs present in urls_list but not in scopus_urls_missing_EID_cited_by: {len(unique_refeids)}")
print("\nFirst 5 URLs from urls_list that are not in scopus_urls_missing_EID_cited_by:")
for url in urls_not_in_scopus_list[:5]:
    print(url)


In [ ]:
import numpy as np

# Split the list into 5 chunks
chunks = np.array_split(urls_not_in_scopus_list, 5)

# Save each chunk to a separate .txt file
for i, chunk in enumerate(chunks):
    file_name = f'urls_not_in_scopus_list_{i+1}.txt'
    with open(file_name, 'w') as f:
        for url in chunk:
            f.write(url + '\n')
    print(f"Saved {len(chunk)} URLs to {file_name}")

Saved 964 URLs to urls_not_in_scopus_list_1.txt
Saved 964 URLs to urls_not_in_scopus_list_2.txt
Saved 963 URLs to urls_not_in_scopus_list_3.txt
Saved 963 URLs to urls_not_in_scopus_list_4.txt
Saved 963 URLs to urls_not_in_scopus_list_5.txt


In [ ]:
import pickle

file_path_pkl = 'scopus_urls_missing_EID_cited_by_list.pkl'

try:
    with open(file_path_pkl, 'wb') as f:
        pickle.dump(scopus_urls_missing_EID_cited_by, f)
    print(f"Successfully saved '{file_path_pkl}' as a list.")
except Exception as e:
    print(f"An error occurred while saving the file '{file_path_pkl}': {e}")

Successfully saved 'scopus_urls_missing_EID_cited_by_list.pkl' as a list.


In [ ]:
json_all_data = scopus_urls_missing_EID_cited_by

In [ ]:
len(json_all_data)

40378

In [ ]:
len(terms_with_many_results)

8791

In [ ]:
# IDENTIFY REFEID OF PAPERS THAT HAVE BEEN CITED MORE THAN 200 TIMES
# cycle through json_all_data and get count in json_all_data[i]['search-results']['opensearch:totalResults'] per  json_all_data[i]

total_results_counts = []
for i in range(len(json_all_data)):
    try:
        count = int(json_all_data[i]['search-results']['opensearch:totalResults'])
        total_results_counts.append(count)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not extract count for item at index {i}: {e}")
        total_results_counts.append(None) # Append None or handle the error as needed

# You can now use the total_results_counts list, for example:
print("Total results counts per JSON object:")
total_results_counts

# in json_all_data, identify all json_all_data[i]['search-results']['opensearch:Query']['@searchTerms'] for which json_all_data[i]['search-results']['opensearch:totalResults'] is larger than 199

terms_with_many_results = []
for i in range(len(json_all_data)):
    try:
        total_results = int(json_all_data[i]['search-results']['opensearch:totalResults'])
        search_terms = json_all_data[i]['search-results']['opensearch:Query']['@searchTerms']
        if total_results > 199:
            terms_with_many_results.append(search_terms)
    except (KeyError, ValueError, TypeError) as e:
        print(f"Could not process item at index {i}: {e}")

print("Search terms with more than 199 total results:")
terms_with_many_results

# Prepare df to make a file for bulk-download of papers that have been cited more than 200 times

table_data = [] # Create an empty list to store data for the table

for i in range(len(terms_with_many_results)):
    search_term = terms_with_many_results[i]
    matching_data = find_matching_data(json_all_data, search_term)
    total_results_str = matching_data['search-results']['opensearch:totalResults']
    total_results = int(total_results_str)
    rounded_up_value = int(np.ceil(total_results / 25))
    table_data.append({'search_term': search_term, 'total_results': total_results, 'rounded_up_value': rounded_up_value})


# After the loop, create the DataFrame from the list of dictionaries
table = pd.DataFrame(table_data)
table['search_term'] = table['search_term'].str.replace('REFEID', '').str.replace('(', '').str.replace(')', '')

expanded_rows = []

for index, row in table.iterrows():
    search_term = row['search_term']
    rounded_up_value = row['rounded_up_value']
    for i in range(rounded_up_value):
        expanded_rows.append({'search_term': search_term, 'rounded_up_value': rounded_up_value})

expanded_table = pd.DataFrame(expanded_rows)
expanded_table = expanded_table.assign(rounded_up_value=expanded_table.groupby('search_term').cumcount())

# Make a list of URLs based on a list of scopus_code

# COMPLETE VIEW - 25 CITATIONS PER PAGE

# Assuming FPE_core is your DataFrame and your API key is 'your_api_key'
api_key = 'XYZ' # Replace with your actual API key
scopus_links = get_scopus_urls_COMPLETE(expanded_table, api_key)
print(scopus_links[:5]) # Print the first 5 links as a sample

In [ ]:
expanded_table

,search_term,rounded_up_value
0,2-s2.0-0006407254,0
1,2-s2.0-0006407254,1
2,2-s2.0-0006407254,2
3,2-s2.0-0006407254,3
4,2-s2.0-0006407254,4
...,...,...
414457,2-s2.0-85068938560,17
414458,2-s2.0-85068938560,18
414459,2-s2.0-85068938560,19
414460,2-s2.0-85068938560,20


In [ ]:
# This creates a txt file, one row per query, which you can export and then quesry SCOPUS per JSON

In [ ]:
api_key = 'XYZ' # Replace with your actual API key
scopus_links = get_scopus_urls_COMPLETE(expanded_table, api_key)
print(scopus_links[:5]) # Print the first 5 links as a sample

In [ ]:
scopus_links_df = pd.DataFrame({'scopus_links': scopus_links})
scopus_links_df.to_csv('scopus_urls_missing_EID_cited_by_long_complete.txt', sep='\t', index=False, header=False)

In [ ]:
scopus_urls_missing_EID_cited_by_long_complete = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Forest_policy_review/scopus_urls_missing_EID_cited_by_long_complete.txt', sep='\t', header=None)

In [ ]:
len(scopus_urls_missing_EID_cited_by_long_complete)

344134

In [ ]:
scopus_links_df_270k_to_end = scopus_urls_missing_EID_cited_by_long_complete.iloc[270000:]
scopus_links_df_270k_to_end

,0
270000,https://api.elsevier.com/content/search/scopus...
270001,https://api.elsevier.com/content/search/scopus...
270002,https://api.elsevier.com/content/search/scopus...
270003,https://api.elsevier.com/content/search/scopus...
270004,https://api.elsevier.com/content/search/scopus...
...,...
344129,https://api.elsevier.com/content/search/scopus...
344130,https://api.elsevier.com/content/search/scopus...
344131,https://api.elsevier.com/content/search/scopus...
344132,https://api.elsevier.com/content/search/scopus...


In [ ]:
%pwd

'/content/drive/MyDrive/Colab Notebooks/Forest_policy_review'

In [ ]:
# Split the DataFrame into 6 equal-sized chunks
chunks = np.array_split(scopus_links_df_270k_to_end, 5)

# Save each chunk to a separate text file
for i, chunk in enumerate(chunks):
    file_name = f'scopus_links_df_270k_to_end{i+1}.txt'
    chunk.to_csv(file_name, index=False, header=False)
    print(f"Saved {len(chunk)} URLs to {file_name}")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Saved 14827 URLs to scopus_links_df_270k_to_end1.txt
Saved 14827 URLs to scopus_links_df_270k_to_end2.txt
Saved 14827 URLs to scopus_links_df_270k_to_end3.txt
Saved 14827 URLs to scopus_links_df_270k_to_end4.txt
Saved 14826 URLs to scopus_links_df_270k_to_end5.txt
